# MineToday V133 — Exact-V132 Support-Gated Three-Expert Utility Consensus

**Frozen experiment contract.** This notebook implements the agreed Post-V132 V133 architecture only. It protects exact public-scoring V132 bytes, reconstructs V132 authority with full third-level nesting, then fits three independent alpha-10 Ridge expected-utility experts (sparse TF-IDF, full-history BM25, and state/readiness). Actions require positive equal-mean consensus, at least 2/3 positive experts, pair×rank support ≥10, and at most three test Top1 swaps. Pseudo-GT is not used.

The registered OOF policy is 12 actions (0.3%); 20/40 are diagnostics only. Any correctness or promotion-gate failure writes exact V132 as the primary submission.


In [1]:
# =========================
# 0. Runtime / dependencies / frozen V133 configuration
# =========================
import os, sys, re, io, gc, json, ast, html, math, time, base64, gzip, hashlib, random, warnings, unicodedata, subprocess, platform
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, Counter

os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("OMP_NUM_THREADS","4")
os.environ.setdefault("OPENBLAS_NUM_THREADS","4")
os.environ.setdefault("MKL_NUM_THREADS","4")
os.environ.setdefault("NUMEXPR_NUM_THREADS","4")
os.environ.setdefault("PYTHONHASHSEED","20260907")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY","1")
warnings.filterwarnings("ignore")

def _ensure_runtime():
    import importlib.metadata as im
    need=[]
    try: im.version("lightgbm")
    except Exception: need.append("lightgbm==4.6.0")
    try:
        tv=im.version("transformers")
        if int(tv.split('.')[0])>=5: need += ["transformers==4.57.1","sentence-transformers==5.1.1"]
    except Exception: need += ["transformers==4.57.1","sentence-transformers==5.1.1"]
    try: im.version("sentence-transformers")
    except Exception:
        if not any(x.startswith("sentence-transformers") for x in need): need.append("sentence-transformers==5.1.1")
    try: im.version("einops")
    except Exception: need.append("einops>=0.8.0")
    if need:
        print("Installing compatibility dependencies before model imports:",need)
        subprocess.check_call([sys.executable,"-m","pip","install","-q",*need])
_ensure_runtime()

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.sparse import hstack, csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
from IPython.display import display

SEED=20260907
random.seed(SEED); np.random.seed(SEED)

@dataclass(frozen=True)
class CFG:
    seed:int=SEED
    n_outer:int=5
    n_inner:int=3
    word_max_features:int=60000
    char_max_features:int=80000
    near_threshold:float=0.90
    lgb_estimators:int=160
    lgb_lr:float=0.03
    lgb_num_leaves:int=15
    lgb_max_depth:int=4
    lgb_min_child:int=80
    lgb_subsample:float=0.80
    lgb_colsample:float=0.75
    lgb_reg_alpha:float=0.5
    lgb_reg_lambda:float=8.0
    lgb_seeds:tuple=(20260907,20260917,20260927)
    action_alpha:float=10.0
    pair_rank_shrink:float=20.0
    v129_coverage:float=0.05
    v130_coverage:float=0.02
    v132_coverage:float=0.005
    v133_support_min:int=10
    v133_primary_actions:int=12
    v133_sensitivity_actions:tuple=(20,40)
    v133_test_cap:int=3
    submit_gain:float=0.00015
    min_positive_folds:int=4
    min_benefit:int=8
    max_harm:int=4
    max_smd:float=1.50
    bootstraps:int=2000
    smoke:bool=bool(int(os.getenv("V133_SMOKE","0")))

cfg=CFG()
if cfg.smoke:
    cfg=CFG(n_outer=2,n_inner=2,word_max_features=8000,char_max_features=12000,lgb_estimators=30,
            lgb_seeds=(20260907,),bootstraps=100,smoke=True)
OUT=Path("/kaggle/working/v133") if Path("/kaggle/working").exists() else Path("./v133")
OUT.mkdir(parents=True,exist_ok=True)
print("CFG",asdict(cfg))
print("numpy",np.__version__,"pandas",pd.__version__,"scipy",scipy.__version__,"sklearn",__import__('sklearn').__version__,"lightgbm",lgb.__version__)
try: print(subprocess.check_output(["nvidia-smi","-L"],text=True).strip())
except Exception: print("GPU not visible; full V133 targets Kaggle T4x2.")


Installing compatibility dependencies before model imports: ['transformers==4.57.1', 'sentence-transformers==5.1.1']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.1 MB/s eta 0:00:00
CFG {'seed': 20260907, 'n_outer': 5, 'n_inner': 3, 'word_max_features': 60000, 'char_max_features': 80000, 'near_threshold': 0.9, 'lgb_estimators': 160, 'lgb_lr': 0.03, 'lgb_num_leaves': 15, 'lgb_max_depth': 4, 'lgb_min_child': 80, 'lgb_subsample': 0.8, 'lgb_colsample': 0.75, 'lgb_reg_alpha': 0.5, 'lgb_reg_lambda': 8.0, 'lgb_seeds': (20260907, 20260917, 20260927), 'action_alpha': 10.0, 'pair_rank_shrink': 20.0, 'v129_coverage': 0.05, 'v130_coverage': 0.02, 'v132_coverage': 0.005, 'v133_support_min': 10, 'v133_primary_actions': 12, 'v133_sensitivity_actions

In [2]:
# =========================
# 1. Organizer data + exact public-positive V132 protected base
# =========================
V132_BASE_SHA256="d89714241c85dbaf93a5574550a5d9272c37b9b432c69fd5c5a2ff07fe690d84"
V132_PUBLIC_LB=0.66219
V132_BASE_GZ_B64="""..."""
NON_S0_TEST_USERS=set(['U_4005', 'U_4015', 'U_4017', 'U_4043', 'U_4044', 'U_4058', 'U_4068', 'U_4069', 'U_4070', 'U_4075', 'U_4087', 'U_4089', 'U_4091', 'U_4098', 'U_4105', 'U_4118', 'U_4121', 'U_4122', 'U_4123', 'U_4130', 'U_4131', 'U_4137', 'U_4138', 'U_4169', 'U_4178', 'U_4183', 'U_4186', 'U_4203', 'U_4214', 'U_4219', 'U_4232', 'U_4233', 'U_4237', 'U_4265', 'U_4266', 'U_4269', 'U_4274', 'U_4285', 'U_4289', 'U_4300', 'U_4304', 'U_4320', 'U_4335', 'U_4345', 'U_4354', 'U_4363', 'U_4379', 'U_4389', 'U_4394', 'U_4402', 'U_4407', 'U_4430', 'U_4436', 'U_4438', 'U_4442', 'U_4480', 'U_4515', 'U_4527', 'U_4532', 'U_4542', 'U_4545', 'U_4555', 'U_4569', 'U_4580', 'U_4582', 'U_4596', 'U_4601', 'U_4633', 'U_4639', 'U_4650', 'U_4661', 'U_4663', 'U_4665', 'U_4672', 'U_4675', 'U_4687', 'U_4717', 'U_4718', 'U_4725', 'U_4730', 'U_4751', 'U_4759', 'U_4763', 'U_4827', 'U_4828', 'U_4838', 'U_4886', 'U_4887', 'U_4893', 'U_4896', 'U_4897', 'U_4905', 'U_4919', 'U_4933', 'U_4942', 'U_4951', 'U_4956', 'U_4960', 'U_4965', 'U_4971', 'U_4978', 'U_4988', 'U_4989'])
V25_PROTECTED_TEST_USERS=set(['U_4047', 'U_4953', 'U_4074', 'U_4176', 'U_4491', 'U_4567', 'U_4836', 'U_4929'])
V132_PROTECTED_TEST_USERS=set(['U_4027', 'U_4028', 'U_4062', 'U_4071', 'U_4128', 'U_4151', 'U_4153', 'U_4155', 'U_4190', 'U_4197', 'U_4200', 'U_4213', 'U_4224', 'U_4246', 'U_4281', 'U_4309', 'U_4329', 'U_4331', 'U_4356', 'U_4397', 'U_4405', 'U_4412', 'U_4415', 'U_4417', 'U_4434', 'U_4454', 'U_4462', 'U_4478', 'U_4490', 'U_4492', 'U_4501', 'U_4508', 'U_4512', 'U_4514', 'U_4524', 'U_4549', 'U_4598', 'U_4617', 'U_4619', 'U_4624', 'U_4626', 'U_4642', 'U_4644', 'U_4654', 'U_4656', 'U_4659', 'U_4664', 'U_4681', 'U_4682', 'U_4703', 'U_4713', 'U_4715', 'U_4720', 'U_4721', 'U_4726', 'U_4731', 'U_4742', 'U_4745', 'U_4780', 'U_4783', 'U_4788', 'U_4794', 'U_4799', 'U_4805', 'U_4807', 'U_4808', 'U_4840', 'U_4843', 'U_4844', 'U_4868', 'U_4872', 'U_4898', 'U_4936', 'U_4979', 'U_4985'])
V131_SELECTED_TEST_IDS=frozenset(['U_4136', 'U_4197', 'U_4590', 'U_4628', 'U_4913'])
assert len(NON_S0_TEST_USERS)==103 and len(V25_PROTECTED_TEST_USERS)==8 and len(V132_PROTECTED_TEST_USERS)==75

REQUIRED={"chat_history.csv","modules_catalog.csv","user_assessments.csv","train_relevance.csv","test.csv","sample_submission.csv"}
def locate_data():
    roots=[Path("/kaggle/input"),Path("."),Path("/mnt/data")]
    for root in roots:
        if not root.exists(): continue
        cands=[root]
        if root==Path("/kaggle/input") or root.name in {"input","data"}:
            cands += [p for p in root.rglob("*") if p.is_dir()]
        for d in cands:
            try: names={p.name for p in d.iterdir() if p.is_file()}
            except Exception: continue
            if REQUIRED.issubset(names): return d
    raise FileNotFoundError("MineToday organizer CSV directory not found under /kaggle/input.")

DATA=locate_data(); print("DATA",DATA)
chat=pd.read_csv(DATA/"chat_history.csv")
modules=pd.read_csv(DATA/"modules_catalog.csv")
assess=pd.read_csv(DATA/"user_assessments.csv")
train=pd.read_csv(DATA/"train_relevance.csv")
test=pd.read_csv(DATA/"test.csv")
sample=pd.read_csv(DATA/"sample_submission.csv")
MODULES=list(train.columns[1:]); M=len(MODULES); NTR=len(train); NTE=len(test); N=NTR+NTE
assert M==17 and NTR==4000 and NTE==1000
assert MODULES==list(sample.columns[1:])
ALL_USERS=train.user_id.astype(str).tolist()+test.user_id.astype(str).tolist(); UID2I={u:i for i,u in enumerate(ALL_USERS)}
y=train[MODULES].to_numpy(np.float32)
assert np.isfinite(y).all()

TEST_ELIGIBLE=np.array([(u not in NON_S0_TEST_USERS) and (u not in V25_PROTECTED_TEST_USERS) for u in test.user_id.astype(str)],bool)
assert int(TEST_ELIGIBLE.sum())==889
V132_TEST_SELECTED=np.asarray([u in V132_PROTECTED_TEST_USERS for u in test.user_id.astype(str)],bool)
assert int(V132_TEST_SELECTED.sum())==75 and np.all(V132_TEST_SELECTED<=TEST_ELIGIBLE)
V133_TEST_ELIGIBLE=TEST_ELIGIBLE & (~V132_TEST_SELECTED)
assert int(V133_TEST_ELIGIBLE.sum())==814

DISCOUNT=1/np.log2(np.arange(2,7,dtype=np.float64))
def top5_idx(scores):
    s=np.asarray(scores); ix=np.argpartition(-s,4,axis=1)[:,:5]; loc=np.take_along_axis(s,ix,1)
    return np.take_along_axis(ix,np.argsort(-loc,axis=1,kind="stable"),1)
def top5_hash(scores): return hashlib.sha256(top5_idx(scores).astype(np.int16).tobytes()).hexdigest()
def per_user_ndcg(yy,ss):
    ix=top5_idx(ss); dcg=(np.take_along_axis(yy,ix,1)*DISCOUNT).sum(1)
    ideal=np.sort(yy,axis=1)[:,-5:][:,::-1]; idcg=(ideal*DISCOUNT).sum(1)
    return dcg/np.maximum(idcg,1e-12)
def ndcg5(yy,ss): return float(per_user_ndcg(yy,ss).mean())
def stable_idx_hash(a): return hashlib.sha256(np.asarray(a,np.int32).tobytes()).hexdigest()

V132_BASE_BYTES=gzip.decompress(base64.b64decode(V132_BASE_GZ_B64.encode()))
assert hashlib.sha256(V132_BASE_BYTES).hexdigest()==V132_BASE_SHA256
v132_base=pd.read_csv(io.BytesIO(V132_BASE_BYTES))
assert v132_base.columns.tolist()==sample.columns.tolist()
assert v132_base.user_id.astype(str).tolist()==test.user_id.astype(str).tolist()
V132_BASE_SCORES=v132_base[MODULES].to_numpy(np.float32)
assert np.isfinite(V132_BASE_SCORES).all()
print("V132 protected base verified",V132_BASE_SHA256,"public",V132_PUBLIC_LB,"frozen Top1 edits",V132_TEST_SELECTED.sum(),"residual eligible",V133_TEST_ELIGIBLE.sum())


DATA /kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026
V132 protected base verified d89714241c85dbaf93a5574550a5d9272c37b9b432c69fd5c5a2ff07fe690d84 public 0.66219 frozen Top1 edits 75 residual eligible 814


In [3]:

# =========================
# 2. Frozen V42 champion-aligned policy-anchor engine
# =========================
# This block is intentionally isolated conceptually from V129's retrieval anchor.
# It reconstructs the V30/V31 + B34/B45 policy geometry inside arbitrary train/validation splits.
all_users=ALL_USERS; uid2idx=UID2I; assessments=assess
N_MODULES=M; N_TRAIN=NTR; N_TEST=NTE
RIDGE_ALPHA=3.0; REFERENCE_W=0.45
ANCHOR_INNER_FOLDS=2 if cfg.smoke else 4
INTENT_INNER_FOLDS=2 if cfg.smoke else 4
LGB_ESTIMATORS=40 if cfg.smoke else 1200
LGB_EARLY_STOP=5 if cfg.smoke else 80
RANKER_ESTIMATORS=40 if cfg.smoke else 360
RANKER_LEARNING_RATE=0.04 if cfg.smoke else 0.025
ALPHA=0.16
INTENT_PARENT_STRENGTH=40.0; INTENT_CHILD_STRENGTH=30.0
BOOTSTRAPS=100 if cfg.smoke else 1200
EXPECTED_BOUNDARY_COUNTS={"B23":9,"B34":10,"B45":8}

def _seed_from_text(s): return int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:8],16)%(2**31-1)
top5_indices=top5_idx
Y_BIN=(y>0).astype(np.float32)
Y_TOP=np.zeros_like(y,np.float32); Y_TOP[np.arange(NTR),np.argmax(y,axis=1)]=1.0

# Robust assessment parsing — exact V3.1 mechanism
WORD_NUM = {"nol":0, "satu":1, "dua":2, "tiga":3, "empat":4, "lima":5}
SKILL_NAMES = ["python","sql","statistics","eda","ml","ml_eval","dl","genai","business","research"]

A = np.zeros((len(all_users), 10), np.float32)
A_INVALID = np.zeros_like(A)

for row in assessments.itertuples(index=False):
    try:
        d = json.loads(row.assessment_result)
    except Exception:
        d = ast.literal_eval(row.assessment_result)

    vals, invalid = [], []
    for v in d.values():
        sv = str(v).strip().lower()
        if sv in WORD_NUM:
            value, bad = float(WORD_NUM[sv]), 0
        else:
            try:
                value = float(v)
            except Exception:
                value = np.nan
            bad = int((not np.isfinite(value)) or value < 0 or value > 5)
            if bad:
                value = np.nan
        vals.append(value)
        invalid.append(bad)

    vals = np.asarray(vals, float)
    med = float(np.nanmedian(vals)) if np.isfinite(vals).any() else 2.0
    vals = np.where(np.isfinite(vals), vals, med)
    i = uid2idx[row.user_id]
    A[i] = vals
    A_INVALID[i] = invalid

print("invalid assessment cells:", int(A_INVALID.sum()), "/", A_INVALID.size)

# Exact V3.1 dataset-derived topic/style parser
TOPIC_PATTERNS = {
    "excel": r"\b(?:ms\.?\s*excel|excel|pivot(?: table)?|vlookup|hlookup|spreadsheet)\b",
    "python": r"\b(?:python|pyton|pithon|pandas(?:\s*&\s*| dan |\s+)?numpy|pandas|numpy)\b",
    "sql": r"\b(?:sql|escuel|query database|join tabel|database relasional|window function)\b",
    "statistics": r"\b(?:statistik(?:\s*&\s*probabilitas)?|probabilitas|a\s*/?\s*b testing|uji hipotesis|distribusi data)\b",
    "eda": r"\b(?:eda|exploratory data analysis|data cleaning|missing values?|matplotlib|seaborn)\b",
    "ml_eval": r"\b(?:evaluasi model(?: ml)?|metrik model|metric model|evaluasi performa model)\b",
    "mlops": r"\b(?:mlops|deployment model|deploy(?:ment)? model|ci\s*/?\s*cd|monitoring model|serving model|fastapi)\b",
    "cv": r"\b(?:computer vision|cnn|deep learning|deteksi objek|klasifikasi gambar|image classification)\b",
    "nlp": r"\b(?:natural language processing|nlp|analisis sentimen|sentiment analysis|word2vec|tf\s*-?\s*idf|transformer)\b",
    "genai": r"\b(?:generative ai|genai|large language model|llm|rag|chatgpt|chatbot|fine[- ]?tuning model bahasa)\b",
    "ml": r"\b(?:machine learning|bikin model ml|model machine learning|regresi\s*(?:&|dan)\s*klasifikasi|clustering|supervised|unsupervised|hyperparameter)\b",
    "business": r"\b(?:translate masalah bisnis ke data|menerjemahkan masalah bisnis|business(?:-| )to(?:-| )ds|permasalahan bisnis)\b",
    "dashboard": r"\b(?:dashboard|looker studio|tableau|visualisasi interaktif|storytelling data)\b",
    "webscrape": r"\b(?:web scrap(?:e|ing)|scraping|beautifulsoup|selenium|ekstraksi data.*website)\b",
    "git": r"\b(?:version control|git(?:hub|lab)?|repository|repo kode)\b",
    "prompt": r"\b(?:prompt engineering|prompting|merancang prompt|optimasi prompt)\b",
    "automation": r"\b(?:ai automation|otomasi|otomatis|automation|workflow|n8n)\b",
    "nocode": r"\b(?:no[ -]?code|zero coding|tanpa perlu (?:belajar )?(?:coding|koding)|gausah (?:coding|koding)|gak perlu (?:coding|koding)|tanpa (?:coding|koding))\b",
    "career": r"\b(?:portofolio|portfolio|cv|test interview|interview|karir|career|prospek kerja|negosiasi)\b",
}
TOPIC_RX = {k: re.compile(v, re.I) for k,v in TOPIC_PATTERNS.items()}
TOPICS = list(TOPIC_PATTERNS)

NOISE_RX = re.compile(
    r"reset password|email verifikasi|sertifikat kelulusan|pembayaran|cicilan|invoice|"
    r"akun.*(?:error|masalah)|lms.*(?:error|masalah)", re.I
)
STYLE_PATTERNS = {
    "beginner": r"pemula|dari nol|gaptek|belum pernah (?:ngoding|coding|koding)|background(?:nya)? bukan (?:it|data)|banting setir",
    "advanced": r"udah lumayan paham|sudah lumayan paham|stuck|prasyarat|syarat|\breq\b|basic apa|lanjutin",
    "urgent": r"urgent|secepatnya|sefastnya|intensif|seminggu|minggu depan",
    "budget": r"budget|diskon|worth it|cekak|fomo|kompetitor",
    "online": r"online|durasi|kapan buka|mulai minggu|jadwal",
    "nocode_style": r"tanpa perlu.*(?:coding|koding)|gausah.*(?:coding|koding)|gak perlu.*(?:coding|koding)|otomatis.*ai|solusi otomatis",
    "career_style": r"portofolio|portfolio|test interview|prospek kerja|karir|career",
}
STYLE_RX = {k: re.compile(v, re.I) for k,v in STYLE_PATTERNS.items()}
STYLES = list(STYLE_PATTERNS)

def clean_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"[^\w\s<>/&+-]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()

def extract_topics(text):
    return [k for k,rx in TOPIC_RX.items() if rx.search(str(text))]

def make_skeleton(text):
    s = clean_text(text)
    for rx in TOPIC_RX.values():
        s = rx.sub(" <topic> ", s)
    s = re.sub(r"\b\d+\b", " <num> ", s)
    return re.sub(r"\s+", " ", s).strip()

def template_token(skeleton):
    return "tpl_" + hashlib.md5(skeleton.encode("utf-8")).hexdigest()[:10]

chat["timestamp"] = pd.to_datetime(chat["timestamp"], errors="coerce")
chat["clean"] = chat["user_chat_text"].fillna("").map(clean_text)
chat["is_noise"] = chat["user_chat_text"].fillna("").map(lambda x: bool(NOISE_RX.search(str(x))))
chat["topics"] = chat["user_chat_text"].fillna("").map(extract_topics)
chat["skeleton"] = chat["user_chat_text"].fillna("").map(make_skeleton)
for name,rx in STYLE_RX.items():
    chat["sty_"+name] = chat["user_chat_text"].fillna("").map(lambda x,r=rx: int(bool(r.search(str(x)))))

chat = chat.sort_values(["user_id","timestamp","chat_id"]).reset_index(drop=True)
user_chat = {u:g.copy() for u,g in chat.groupby("user_id", sort=False)}
print("administrative/noise rows:", int(chat["is_noise"].sum()), "/", len(chat))

# V3.1 template-aware sparse representation + structured user features
USER_FEATURES = np.zeros((len(all_users), 10 + 10 + 5 + len(STYLES)*2 + len(TOPICS)*5), np.float32)

def build_user(u):
    ui = uid2idx[u]
    g = user_chat.get(u)
    if g is None:
        learn = pd.DataFrame(columns=chat.columns)
        g = learn
    else:
        learn = g[~g["is_noise"]].copy()
        if len(learn) == 0:
            learn = g.copy()

    parts = []
    n = len(learn)
    for j,row in enumerate(learn.itertuples(index=False)):
        distance = n - 1 - j
        tag = "current" if distance == 0 else ("recent" if distance <= 2 else "older")
        parts.append(f"[{tag}] {row.clean}")
        parts.append(template_token(row.skeleton))
        for tp in row.topics:
            parts.append("topic_" + tp)
            if distance == 0:
                parts.extend(["latest_topic_" + tp] * 3)
            elif distance <= 2:
                parts.extend(["recent_topic_" + tp] * 2)
        for sty in STYLES:
            if getattr(row, "sty_"+sty):
                parts.append("style_" + sty)

    for j,skill in enumerate(SKILL_NAMES):
        v = int(round(float(A[ui,j])))
        band = "low" if v <= 1 else ("mid" if v <= 3 else "high")
        parts += [f"skill_{skill}_{v}", f"skill_{skill}_bin_{band}"]

    feats = list(A[ui]) + list(A_INVALID[ui])
    total = len(g)
    learning_count = len(learn)
    noise_count = int(g["is_noise"].sum()) if total else 0
    span = 0.0
    if total >= 2 and g["timestamp"].notna().sum() >= 2:
        span = float((g["timestamp"].max() - g["timestamp"].min()).total_seconds() / 86400.0)
    feats += [total, learning_count, noise_count, span, min(total,6)]

    for sty in STYLES:
        feats.append(float(learn["sty_"+sty].sum()) if learning_count else 0.0)
    for sty in STYLES:
        feats.append(float(learn.iloc[-1]["sty_"+sty]) if learning_count else 0.0)

    rows = list(learn.itertuples(index=False))
    nfull = len(rows)
    for tp in TOPICS:
        count = rec_count = last_pos = 0.0
        latest = 0.0
        for j,row in enumerate(rows):
            if tp in row.topics:
                count += 1.0
                rec_count += float(np.exp(-0.7 * (nfull - 1 - j)))
                last_pos = max(last_pos, (j+1) / max(nfull,1))
        if rows and tp in rows[-1].topics:
            latest = 1.0
        feats += [count, rec_count, latest, last_pos, float(count > 0)]

    USER_FEATURES[ui] = np.asarray(feats, np.float32)
    return " ".join(parts)

texts = np.asarray([build_user(u) for u in all_users], dtype=object)
assert USER_FEATURES.shape == (5000, 134)
assert np.isfinite(USER_FEATURES).all()
print("USER_FEATURES:", USER_FEATURES.shape)
print("example augmented text:", texts[1][:500])


# V42 target-blind sparse text basis. Full 5,000-user vocabulary matches the frozen lineage.
word_vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.995,sublinear_tf=True,strip_accents="unicode",max_features=(8000 if cfg.smoke else 50000),dtype=np.float32)
char_vec=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,sublinear_tf=True,max_features=(12000 if cfg.smoke else 60000),dtype=np.float32)
X_WORD=word_vec.fit_transform(texts); X_CHAR=char_vec.fit_transform(texts); X=hstack([X_WORD,X_CHAR],format="csr")
CONSENSUS3_OVERRIDES={"M_002":0.44,"M_006":0.55,"M_007":0.60,"M_010":0.40,"M_011":0.55,"M_012":0.55,"M_015":0.44,"M_017":0.53}
def consensus3_vector():
    w=np.full(N_MODULES,REFERENCE_W,np.float32)
    for mid,val in CONSENSUS3_OVERRIDES.items(): w[MODULES.index(mid)]=float(val)
    return w
CONS_W=consensus3_vector()

# Module-native lexical / intent / readiness surfaces inherited from V30 Base.
MODULE_TEXT = (
    modules.set_index("module_id").loc[MODULES, "module_name"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "description_and_syllabus"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "prerequisite_level"].astype(str)
).tolist()
MODULE_WORD = word_vec.transform(MODULE_TEXT)
MODULE_CHAR = char_vec.transform(MODULE_TEXT)
LEX_WORD = (X_WORD @ MODULE_WORD.T).toarray().astype(np.float32)
LEX_CHAR = (X_CHAR @ MODULE_CHAR.T).toarray().astype(np.float32)

MODULE_TOPIC_MAP = {
    "M_001":["excel"], "M_002":["python"], "M_003":["sql"],
    "M_004":["webscrape","python"], "M_005":["git"], "M_006":["statistics"],
    "M_007":["eda","python","statistics"], "M_008":["dashboard","sql","excel"],
    "M_009":["ml","ml_eval","eda","statistics"], "M_010":["cv","ml"],
    "M_011":["nlp","ml"], "M_012":["genai","nlp"], "M_013":["prompt","genai"],
    "M_014":["automation","genai"], "M_015":["nocode","genai"],
    "M_016":["mlops","ml","python","git"], "M_017":["career"],
}
MODULE_SKILL_MAP = {
    "M_001":[], "M_002":[0], "M_003":[1], "M_004":[0], "M_005":[9],
    "M_006":[2], "M_007":[0,2,3], "M_008":[1], "M_009":[2,3,4,5],
    "M_010":[4,6], "M_011":[4], "M_012":[4,7], "M_013":[7],
    "M_014":[7,8], "M_015":[7], "M_016":[0,4,5,9], "M_017":[8,9],
}

MODULE_COUNT = np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_RECENCY = np.zeros_like(MODULE_COUNT)
MODULE_LATEST = np.zeros_like(MODULE_COUNT)
LATEST_GROUP = np.empty(len(all_users),dtype=object); LATEST_GROUP[:] = "no_chat"
LATEST_INTENT_KEY = np.empty(len(all_users),dtype=object); LATEST_INTENT_KEY[:] = "intent:no_chat"

for u in all_users:
    ui=uid2idx[u]
    g=user_chat.get(u)
    if g is None or len(g)==0:
        continue
    learn=g[~g["is_noise"]].copy()
    if len(learn)==0:
        learn=g.copy()
    rows=list(learn.itertuples(index=False))
    if rows:
        LATEST_GROUP[ui]=rows[-1].skeleton or "empty"
        tps=sorted(set(rows[-1].topics))
        LATEST_INTENT_KEY[ui] = "intent:" + ("+".join(tps) if tps else "none")
    n=len(rows)
    for j,row in enumerate(rows):
        dist=n-1-j; w=float(np.exp(-0.7*dist)); row_topics=set(row.topics)
        for m,mid in enumerate(MODULES):
            hits=sum(tp in row_topics for tp in MODULE_TOPIC_MAP[mid])
            if hits:
                MODULE_COUNT[ui,m]+=hits
                MODULE_RECENCY[ui,m]+=w*hits
                if dist==0:
                    MODULE_LATEST[ui,m]=1.0

MODULE_READINESS=np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_INVALID=np.zeros_like(MODULE_READINESS)
for m,mid in enumerate(MODULES):
    idx=MODULE_SKILL_MAP[mid]
    if idx:
        MODULE_READINESS[:,m]=A[:,idx].mean(1)
        MODULE_INVALID[:,m]=A_INVALID[:,idx].mean(1)
    else:
        MODULE_READINESS[:,m]=A.mean(1)
        MODULE_INVALID[:,m]=A_INVALID.mean(1)

# Fixed readiness discretization consistent with the V3.1 low/mid/high assessment bins.
_ready_round=np.floor(MODULE_READINESS+0.5)
MODULE_READY_BAND=np.where(_ready_round<=1,0,np.where(_ready_round<=3,1,2)).astype(np.int8)

TRAIN_GROUP=np.asarray(LATEST_GROUP[:N_TRAIN],dtype=object)
TRAIN_INTENT=np.asarray(LATEST_INTENT_KEY[:N_TRAIN],dtype=object)
TEST_INTENT=np.asarray(LATEST_INTENT_KEY[N_TRAIN:],dtype=object)

prtxt=modules.set_index("module_id")["prerequisite_level"].astype(str).to_dict()
def module_level(mid):
    s=prtxt[mid].lower()
    if "pemula" in s: return 0
    if "menengah" in s: return 1
    if "lanjutan" in s: return 2
    if "ahli" in s: return 3
    return 4
MODULE_LEVEL=np.asarray([module_level(m) for m in MODULES],np.int16)
MODULE_LEVEL_ONEHOT=np.eye(5,dtype=np.float32)[MODULE_LEVEL]

# Ordinal LambdaRank labels: 0=irrelevant, 1=#6 ... 6=#1.
ORDINAL_GRADES=np.zeros_like(y,dtype=np.int16)
for i,row in enumerate(y):
    pos=np.flatnonzero(row>0)
    order=pos[np.argsort(-row[pos],kind="mergesort")]
    for rank,m in enumerate(order,1):
        ORDINAL_GRADES[i,m]=max(1,7-rank)
LABEL_GAIN=[0.0,0.25,0.40,0.55,0.70,0.85,1.00]

def user_strata_from_y(yy):
    yy=np.asarray(yy)
    k=(yy>0).sum(1).astype(int)
    top=np.argmax(yy,axis=1).astype(int)
    return (top*4+np.clip(k-3,0,3)).astype(int)

def make_group_folds_local(yy,groups,n_splits,seed):
    yy=np.asarray(yy); groups=np.asarray(groups,dtype=object)
    ns=min(int(n_splits),len(np.unique(groups)))
    if ns<2:
        sk=StratifiedKFold(n_splits=2,shuffle=True,random_state=int(seed))
        return list(sk.split(np.zeros(len(yy)),user_strata_from_y(yy)))
    sg=StratifiedGroupKFold(n_splits=ns,shuffle=True,random_state=int(seed))
    folds=list(sg.split(np.zeros(len(yy)),user_strata_from_y(yy),groups=groups))
    for tr,va in folds:
        assert len(set(groups[tr]).intersection(set(groups[va])))==0
    return folds

def make_group_folds_global(rows,n_splits,seed):
    rows=np.asarray(rows,int)
    local=make_group_folds_local(y[rows],TRAIN_GROUP[rows],n_splits,seed)
    return [(rows[tr],rows[va]) for tr,va in local]

print("catalog lexical matrices:",LEX_WORD.shape,LEX_CHAR.shape)
print("latest template groups:",len(np.unique(TRAIN_GROUP)))
print("latest intent signatures train/test:",len(np.unique(TRAIN_INTENT)),len(np.unique(TEST_INTENT)))
print("test intent signatures unseen in train:",len(set(TEST_INTENT)-set(TRAIN_INTENT)))
print("readiness band counts:",dict(zip(*np.unique(MODULE_READY_BAND[:N_TRAIN],return_counts=True))))
print("ordinal grade counts:",dict(zip(*np.unique(ORDINAL_GRADES,return_counts=True))))

ALL_INTENT=np.concatenate([np.asarray(TRAIN_INTENT,dtype=object),np.asarray(TEST_INTENT,dtype=object)])
# Strict fold-safe hierarchical Intent-State target encoding.
INTENTSTATE_KEYS = [
    "IS_PARENT_SEEN","IS_PARENT_SUPPORT_LOG","IS_PARENT_CONF",
    "IS_PARENT_POS","IS_PARENT_ORD","IS_PARENT_RAW",
    "IS_CHILD_SUPPORT_LOG","IS_CHILD_CONF",
    "IS_CHILD_POS","IS_CHILD_ORD","IS_CHILD_RAW","IS_CHILD_TOP5",
    "IS_POS_LOG_LIFT_GLOBAL","IS_ORD_DELTA_GLOBAL","IS_ORD_DELTA_PARENT","IS_RAW_DELTA_PARENT",
]

def _intent_targets(y_fit,grades_fit):
    y_fit=np.asarray(y_fit,np.float32); grades_fit=np.asarray(grades_fit,np.int16)
    return {
        "pos":(y_fit>0).astype(np.float32),
        "ord":(grades_fit.astype(np.float32)/6.0),
        "raw":y_fit.astype(np.float32),
        "top5":(grades_fit>=2).astype(np.float32),
    }

def fit_intentstate_bundle(y_fit,grades_fit,intent_fit,ready_band_fit):
    y_fit=np.asarray(y_fit,np.float32)
    grades_fit=np.asarray(grades_fit,np.int16)
    intent_fit=np.asarray(intent_fit,dtype=object)
    ready_band_fit=np.asarray(ready_band_fit,np.int8)
    n=len(y_fit)
    assert grades_fit.shape==y_fit.shape==ready_band_fit.shape
    T=_intent_targets(y_fit,grades_fit)
    glob={k:v.mean(0).astype(np.float32) for k,v in T.items()}

    parent={}
    child={}
    for key in np.unique(intent_fit):
        idx=np.flatnonzero(intent_fit==key)
        ns=int(len(idx))
        vals={}
        for name,M in T.items():
            sm=(M[idx].sum(0)+INTENT_PARENT_STRENGTH*glob[name])/(ns+INTENT_PARENT_STRENGTH)
            vals[name]=sm.astype(np.float32)
        parent[str(key)]={"n":ns,**vals}

        for m in range(N_MODULES):
            rb=ready_band_fit[idx,m]
            for b in (0,1,2):
                local=idx[rb==b]
                nc=int(len(local))
                if nc==0:
                    continue
                cv={}
                for name,M in T.items():
                    prior=float(vals[name][m])
                    cv[name]=float((M[local,m].sum()+INTENT_CHILD_STRENGTH*prior)/(nc+INTENT_CHILD_STRENGTH))
                child[(str(key),m,int(b))]={"n":nc,**cv}

    return {"n_fit":n,"global":glob,"parent":parent,"child":child}

def apply_intentstate_bundle(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object)
    ready_band=np.asarray(ready_band,np.int8)
    n=len(intent_keys)
    assert ready_band.shape==(n,N_MODULES)
    out={k:np.zeros((n,N_MODULES),np.float32) for k in INTENTSTATE_KEYS}
    glob=bundle["global"]
    logden=np.log1p(max(1,int(bundle["n_fit"])))

    for i,key0 in enumerate(intent_keys):
        key=str(key0)
        p=bundle["parent"].get(key)
        parent_seen=float(p is not None)
        if p is None:
            pn=0
            pvals={name:glob[name] for name in ("pos","ord","raw","top5")}
        else:
            pn=int(p["n"])
            pvals={name:p[name] for name in ("pos","ord","raw","top5")}
        pconf=pn/(pn+INTENT_PARENT_STRENGTH) if pn else 0.0
        pslog=np.log1p(pn)/logden if logden>0 else 0.0

        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            if c is None:
                cn=0
                cpos=float(pvals["pos"][m]); cord=float(pvals["ord"][m])
                craw=float(pvals["raw"][m]); ctop5=float(pvals["top5"][m])
            else:
                cn=int(c["n"]); cpos=float(c["pos"]); cord=float(c["ord"])
                craw=float(c["raw"]); ctop5=float(c["top5"])
            cconf=cn/(cn+INTENT_CHILD_STRENGTH) if cn else 0.0
            cslog=np.log1p(cn)/logden if logden>0 else 0.0
            gpos=float(glob["pos"][m]); gord=float(glob["ord"][m]); graw=float(glob["raw"][m])
            pord=float(pvals["ord"][m]); praw=float(pvals["raw"][m])

            out["IS_PARENT_SEEN"][i,m]=parent_seen
            out["IS_PARENT_SUPPORT_LOG"][i,m]=pslog
            out["IS_PARENT_CONF"][i,m]=pconf
            out["IS_PARENT_POS"][i,m]=float(pvals["pos"][m])
            out["IS_PARENT_ORD"][i,m]=pord
            out["IS_PARENT_RAW"][i,m]=praw
            out["IS_CHILD_SUPPORT_LOG"][i,m]=cslog
            out["IS_CHILD_CONF"][i,m]=cconf
            out["IS_CHILD_POS"][i,m]=cpos
            out["IS_CHILD_ORD"][i,m]=cord
            out["IS_CHILD_RAW"][i,m]=craw
            out["IS_CHILD_TOP5"][i,m]=ctop5
            out["IS_POS_LOG_LIFT_GLOBAL"][i,m]=float(np.clip(np.log((cpos+1e-4)/(gpos+1e-4)),-2.0,2.0))
            out["IS_ORD_DELTA_GLOBAL"][i,m]=cord-gord
            out["IS_ORD_DELTA_PARENT"][i,m]=cord-pord
            out["IS_RAW_DELTA_PARENT"][i,m]=craw-praw

    for k,v in out.items():
        if not np.isfinite(v).all():
            raise ValueError(f"non-finite Intent-State feature {k}")
    return out

def crossfit_intentstate_features(y_local,grades_local,intent_local,ready_band_local,groups_local,seed):
    y_local=np.asarray(y_local,np.float32)
    grades_local=np.asarray(grades_local,np.int16)
    intent_local=np.asarray(intent_local,dtype=object)
    ready_band_local=np.asarray(ready_band_local,np.int8)
    groups_local=np.asarray(groups_local,dtype=object)
    out={k:np.zeros_like(y_local,np.float32) for k in INTENTSTATE_KEYS}
    folds=make_group_folds_local(y_local,groups_local,INTENT_INNER_FOLDS,seed)
    seen=np.zeros(len(y_local),bool)
    for tr,va in folds:
        assert len(set(groups_local[tr]).intersection(set(groups_local[va])))==0
        bundle=fit_intentstate_bundle(y_local[tr],grades_local[tr],intent_local[tr],ready_band_local[tr])
        vv=apply_intentstate_bundle(bundle,intent_local[va],ready_band_local[va])
        for k in INTENTSTATE_KEYS:
            out[k][va]=vv[k]
        seen[va]=True
    assert seen.all(), "Intent-State crossfit did not cover every training row"
    return out

def intentstate_support_audit(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object); ready_band=np.asarray(ready_band,np.int8)
    supports=[]; parent_support=[]; parent_seen=[]
    for i,key0 in enumerate(intent_keys):
        key=str(key0); p=bundle["parent"].get(key)
        pn=int(p["n"]) if p is not None else 0
        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            supports.append(int(c["n"]) if c is not None else 0)
            parent_support.append(pn); parent_seen.append(int(p is not None))
    s=np.asarray(supports,int); ps=np.asarray(parent_support,int); pseen=np.asarray(parent_seen,int)
    rows=[]
    for thr in [1,5,10,20,30,50]:
        rows.append({"level":"child","threshold":thr,"coverage":float(np.mean(s>=thr))})
    rows.append({"level":"child","threshold":-1,"coverage":1.0,"median_support":float(np.median(s)),"p10_support":float(np.quantile(s,.10)),"p25_support":float(np.quantile(s,.25))})
    rows.append({"level":"parent","threshold":1,"coverage":float(np.mean(pseen>0)),"median_support":float(np.median(ps))})
    return pd.DataFrame(rows)

print("strict nested Intent-State TE helpers ready; features:",len(INTENTSTATE_KEYS))

def rank_matrix(scores):
    order=np.argsort(-np.asarray(scores),axis=1,kind="mergesort")
    ranks=np.empty_like(order,dtype=np.int16)
    ranks[np.arange(len(order))[:,None],order]=np.arange(1,N_MODULES+1,dtype=np.int16)[None,:]
    return ranks

def top5_order_change_rate(a,b):
    return float(np.mean(np.any(top5_indices(a)!=top5_indices(b),axis=1)))

def build_long_matrix_v31(rows_global,anchor,base,meta,intentstate=None):
    rows_global=np.asarray(rows_global,int); anchor=np.asarray(anchor,np.float32)
    base=np.asarray(base,np.float32); meta=np.asarray(meta,np.float32); n=len(rows_global)

    ranks=rank_matrix(anchor).astype(np.float32)
    top1=anchor.max(1,keepdims=True)
    sorted_scores=np.sort(anchor,axis=1)[:,::-1]
    cut5=sorted_scores[:,4:5]
    mu=anchor.mean(1,keepdims=True); sd=np.maximum(anchor.std(1,keepdims=True),1e-4)
    z=(anchor-mu)/sd
    order=np.argsort(-anchor,axis=1,kind="mergesort")
    prev_gap=np.zeros_like(anchor); next_gap=np.zeros_like(anchor)
    for i in range(n):
        oo=order[i]; ss=anchor[i,oo]
        pg=np.r_[0.0,ss[:-1]-ss[1:]]
        ng=np.r_[ss[:-1]-ss[1:],0.0]
        prev_gap[i,oo]=pg; next_gap[i,oo]=ng

    base_mats=[
        anchor, ranks/float(N_MODULES), (ranks<=5).astype(np.float32), top1-anchor, anchor-cut5,
        prev_gap,next_gap,z,base,meta,meta-base,
        LEX_WORD[rows_global],LEX_CHAR[rows_global],MODULE_COUNT[rows_global],MODULE_RECENCY[rows_global],
        MODULE_LATEST[rows_global],MODULE_READINESS[rows_global],MODULE_INVALID[rows_global],
    ]
    base_names=[
        "anchor_score","anchor_rank_norm","anchor_top5","top1_minus_score","score_minus_cut5",
        "prev_gap","next_gap","anchor_z","base_score","meta_score","meta_minus_base",
        "lex_word","lex_char","topic_count","topic_recency","topic_latest","readiness","invalid_readiness",
    ]
    pieces=[np.column_stack([m.reshape(-1) for m in base_mats]).astype(np.float32)]
    names=list(base_names)

    pieces.append(np.repeat(USER_FEATURES[rows_global],N_MODULES,axis=0).astype(np.float32))
    names += [f"user_f{i:03d}" for i in range(USER_FEATURES.shape[1])]
    pieces.append(np.tile(np.eye(N_MODULES,dtype=np.float32),(n,1)))
    names += [f"module_{m}" for m in MODULES]
    pieces.append(np.tile(MODULE_LEVEL_ONEHOT,(n,1)).astype(np.float32))
    names += [f"module_level_{i}" for i in range(5)]
    bounds={"base":sum(p.shape[1] for p in pieces)}

    if intentstate is not None:
        mats=[intentstate[k] for k in INTENTSTATE_KEYS]
        pieces.append(np.column_stack([m.reshape(-1) for m in mats]).astype(np.float32))
        names += [k.lower() for k in INTENTSTATE_KEYS]
        bounds["intentstate"]=sum(p.shape[1] for p in pieces)

    XX=np.hstack(pieces).astype(np.float32,copy=False)
    if XX.shape[0] != n*N_MODULES:
        raise AssertionError(XX.shape)
    if len(names)!=XX.shape[1]:
        raise AssertionError((len(names),XX.shape))
    if not np.isfinite(XX).all():
        raise ValueError("non-finite V31 long features")
    return XX,names,bounds

def make_ranker(seed):
    return lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=RANKER_ESTIMATORS,
        learning_rate=RANKER_LEARNING_RATE,
        num_leaves=15,
        min_child_samples=80,
        reg_alpha=1.0,
        reg_lambda=10.0,
        colsample_bytree=0.78,
        subsample=0.90,
        subsample_freq=1,
        max_bin=127,
        lambdarank_truncation_level=8,
        label_gain=LABEL_GAIN,
        random_state=int(seed),
        n_jobs=4,
        verbosity=-1,
        force_col_wise=True,
    )

def fit_ranker_predict(Xtr,Xva,grades_tr,n_users,seed):
    model=make_ranker(seed)
    group=np.full(int(n_users),N_MODULES,dtype=np.int32)
    model.fit(Xtr,grades_tr.reshape(-1).astype(np.int32),group=group)
    pred=model.predict(Xva).reshape(-1,N_MODULES).astype(np.float32)
    return model,pred

def align_ranker_to_anchor(raw_ranker,anchor):
    raw=np.asarray(raw_ranker,np.float32); anchor=np.asarray(anchor,np.float32)
    rmu=raw.mean(1,keepdims=True); rsd=np.maximum(raw.std(1,keepdims=True),1e-5)
    amu=anchor.mean(1,keepdims=True); asd=np.maximum(anchor.std(1,keepdims=True),1e-5)
    return (amu+(raw-rmu)*(asd/rsd)).astype(np.float32)

def blend_residual(anchor,raw_ranker,alpha=ALPHA):
    aligned=align_ranker_to_anchor(raw_ranker,anchor)
    return (anchor+float(alpha)*(aligned-anchor)).astype(np.float32)

def cluster_boot_ci(values,groups,seed,B=BOOTSTRAPS):
    values=np.asarray(values,float); groups=np.asarray(groups,dtype=object)
    ug=np.unique(groups); by={g:np.flatnonzero(groups==g) for g in ug}
    rng=np.random.default_rng(int(seed)); z=np.empty(int(B),float)
    for b in range(int(B)):
        sampled=rng.choice(ug,size=len(ug),replace=True)
        idx=np.concatenate([by[g] for g in sampled])
        z[b]=values[idx].mean()
    return tuple(float(x) for x in np.quantile(z,[.025,.975]))

print("V30-control / V31-IntentState LambdaRank helpers ready")

def _make_cal_local(seed,n_estimators=None):
    return lgb.LGBMClassifier(
        n_estimators=int(LGB_ESTIMATORS if n_estimators is None else n_estimators),
        learning_rate=.03,num_leaves=15,min_child_samples=50,
        reg_alpha=.5,reg_lambda=5.,colsample_bytree=.7,
        random_state=int(seed),n_jobs=4,verbosity=-1
    )


def build_anchor_split_full(train_rows,val_rows,seed):
    train_rows=np.asarray(train_rows,int); val_rows=np.asarray(val_rows,int)
    local_y=y[train_rows]; local_top=Y_TOP[train_rows]; local_bin=Y_BIN[train_rows]
    folds=make_group_folds_local(local_y,TRAIN_GROUP[train_rows],ANCHOR_INNER_FOLDS,seed)
    n=len(train_rows)

    raw=np.zeros((n,N_MODULES),np.float32); top=np.zeros_like(raw)
    for lt,lv in folds:
        mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
        mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
        mr.fit(X[train_rows[lt]],local_y[lt]); mt.fit(X[train_rows[lt]],local_top[lt])
        raw[lv]=mr.predict(X[train_rows[lv]]).astype(np.float32)
        top[lv]=mt.predict(X[train_rows[lv]]).astype(np.float32)
    mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
    mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
    mr.fit(X[train_rows],local_y); mt.fit(X[train_rows],local_top)
    base_o=(.9*raw+.1*top).astype(np.float32)
    base_v=(.9*mr.predict(X[val_rows])+.1*mt.predict(X[val_rows])).astype(np.float32)

    meta_o=np.zeros_like(base_o); meta_v=np.zeros((len(val_rows),N_MODULES),np.float32)
    for m,mid in enumerate(MODULES):
        xt=np.column_stack([base_o[:,m],USER_FEATURES[train_rows]]).astype(np.float32)
        xv=np.column_stack([base_v[:,m],USER_FEATURES[val_rows]]).astype(np.float32)
        yy=local_bin[:,m].astype(np.int32); its=[]
        for f,(lt,lv) in enumerate(folds):
            clf=_make_cal_local(seed+31*m+f)
            clf.fit(xt[lt],yy[lt],eval_set=[(xt[lv],yy[lv])],callbacks=[lgb.early_stopping(LGB_EARLY_STOP,verbose=False)])
            it=int(clf.best_iteration_ or LGB_ESTIMATORS); its.append(it)
            meta_o[lv,m]=clf.predict_proba(xt[lv],num_iteration=it)[:,1].astype(np.float32)
        final_it=max(20,int(np.median(its)))
        full=_make_cal_local(seed+997+31*m,final_it); full.fit(xt,yy)
        meta_v[:,m]=full.predict_proba(xv)[:,1].astype(np.float32)

    cons_o=((1.0-CONS_W)*base_o+CONS_W*meta_o).astype(np.float32)
    cons_v=((1.0-CONS_W)*base_v+CONS_W*meta_v).astype(np.float32)
    return {"base_train":base_o,"meta_train":meta_o,"cons_train":cons_o,
            "base_val":base_v,"meta_val":meta_v,"cons_val":cons_v}

print("Nested frozen anchor helper ready")

STATE_NAMES=np.asarray(["S0","S1","S2","S3"],dtype=object)


def state_codes(pre_anchor,v30,v31):
    pre_top=top5_indices(pre_anchor); v30_top=top5_indices(v30); v31_top=top5_indices(v31)
    a=np.any(pre_top!=v30_top,axis=1)
    b=np.any(v30_top!=v31_top,axis=1)
    return (2*a.astype(np.int8)+b.astype(np.int8)).astype(np.int8)


def top5_sha256(scores):
    top=top5_indices(scores).astype(np.int16)
    return hashlib.sha256(top.tobytes()).hexdigest()


def current066020_surface(v30,v31,state):
    v30=np.asarray(v30,np.float32); v31=np.asarray(v31,np.float32); state=np.asarray(state,np.int8)
    t30=top5_indices(v30); t31=top5_indices(v31)
    top1=(t30[:,0]!=t31[:,0])
    membership=np.asarray([set(a)!=set(b) for a,b in zip(t30,t31)],bool)
    s3=(state==3)
    keep=s3 & (top1 | membership)
    out=v30.copy(); out[keep]=v31[keep]
    info={
        "s3":s3,
        "top1":s3 & top1,
        "membership":s3 & membership & ~top1,
        "order_only":s3 & ~top1 & ~membership,
        "accepted":keep,
    }
    return out,info


def classify_single_adjacent_order_swap(v30_top_row,v31_top_row):
    a=np.asarray(v30_top_row,int); b=np.asarray(v31_top_row,int)
    if a[0]!=b[0] or set(a)!=set(b):
        raise AssertionError(("not pure order-only",a.tolist(),b.tolist()))
    diff=np.flatnonzero(a!=b)
    if len(diff)!=2 or diff[1]!=diff[0]+1:
        raise AssertionError(("not a single adjacent transposition",a.tolist(),b.tolist(),diff.tolist()))
    lo,hi=int(diff[0]),int(diff[1])
    if not (a[lo]==b[hi] and a[hi]==b[lo]):
        raise AssertionError(("not a two-way swap",a.tolist(),b.tolist()))
    boundary=f"B{lo+1}{hi+1}"
    if boundary not in EXPECTED_BOUNDARY_COUNTS:
        raise AssertionError(("unexpected order-only boundary",boundary,a.tolist(),b.tolist()))
    promoted=int(b[lo])
    demoted=int(a[lo])
    return boundary,promoted,demoted,lo+1,hi+1


def submission_from_scores(scores):
    out=sample.copy(); out.loc[:,MODULES]=np.asarray(scores,np.float32)
    assert out.shape==sample.shape
    assert out["user_id"].astype(str).tolist()==test["user_id"].astype(str).tolist()
    assert np.isfinite(out[MODULES].to_numpy()).all()
    return out


def candidate_geometry(name,anchor,candidate):
    ta=top5_indices(anchor); tc=top5_indices(candidate)
    changed=np.any(ta!=tc,axis=1)
    membership=np.asarray([set(a)!=set(b) for a,b in zip(ta,tc)],bool)
    return {
        "candidate":name,
        "changed_users_vs_current":int(changed.sum()),
        "top1_changes":int(np.sum(ta[:,0]!=tc[:,0])),
        "membership_changes":int(membership.sum()),
        "top5_sha256":top5_sha256(candidate),
    }

print("V39.5 state machine + causal boundary helpers ready")


def build_policy_anchor_split(train_rows, val_rows, seed):
    """Fit the frozen structural lineage only on train_rows and predict arbitrary global val_rows."""
    train_rows = np.asarray(train_rows, int)
    val_rows = np.asarray(val_rows, int)
    anc = build_anchor_split_full(train_rows, val_rows, int(seed))

    # Intent-State training features are strictly cross-fitted inside train_rows.
    intent_tr = crossfit_intentstate_features(
        y[train_rows], ORDINAL_GRADES[train_rows], TRAIN_INTENT[train_rows],
        MODULE_READY_BAND[train_rows], TRAIN_GROUP[train_rows], int(seed)+71
    )
    bundle = fit_intentstate_bundle(
        y[train_rows], ORDINAL_GRADES[train_rows], TRAIN_INTENT[train_rows], MODULE_READY_BAND[train_rows]
    )
    intent_va = apply_intentstate_bundle(bundle, ALL_INTENT[val_rows], MODULE_READY_BAND[val_rows])

    Xtr, names, bounds = build_long_matrix_v31(
        train_rows, anc["cons_train"], anc["base_train"], anc["meta_train"], intent_tr
    )
    Xva, _, bounds2 = build_long_matrix_v31(
        val_rows, anc["cons_val"], anc["base_val"], anc["meta_val"], intent_va
    )
    assert bounds == bounds2
    be = bounds["base"]
    ie = bounds["intentstate"]
    _, raw30 = fit_ranker_predict(Xtr[:,:be], Xva[:,:be], ORDINAL_GRADES[train_rows], len(train_rows), int(seed)+101)
    _, raw31 = fit_ranker_predict(Xtr[:,:ie], Xva[:,:ie], ORDINAL_GRADES[train_rows], len(train_rows), int(seed)+102)
    v30 = blend_residual(anc["cons_val"], raw30, ALPHA)
    v31 = blend_residual(anc["cons_val"], raw31, ALPHA)
    st = state_codes(anc["cons_val"], v30, v31)
    current, info = current066020_surface(v30, v31, st)

    # Apply the public-confirmed B34+B45 family mechanically, never by user id.
    t30 = top5_indices(v30); t31 = top5_indices(v31)
    for r in np.flatnonzero(info["order_only"]):
        try:
            boundary, *_ = classify_single_adjacent_order_swap(t30[r], t31[r])
        except AssertionError:
            continue
        if boundary in {"B34", "B45"}:
            current[r] = v31[r]

    # Training structural surface remains purely cross-fitted inside train_rows.
    train_anchor = anc["cons_train"].copy()
    out = {
        "train_anchor": train_anchor,
        "val_anchor": current.astype(np.float32),
        "val_state": st.astype(np.int8),
        "base_train": anc["base_train"], "meta_train": anc["meta_train"],
        "base_val": anc["base_val"], "meta_val": anc["meta_val"],
    }
    del Xtr, Xva, intent_tr, intent_va, bundle, anc, raw30, raw31
    gc.collect()
    return out

print("policy-matched validation anchor helper ready")

print("V42 policy engine initialized; explicit policy_anchor role ready")

POLICY_CACHE=OUT/"policy_anchor_cache"; POLICY_CACHE.mkdir(parents=True,exist_ok=True)
def cached_policy_anchor_split(train_rows,val_rows,seed):
    tr=np.asarray(train_rows,np.int32); va=np.asarray(val_rows,np.int32)
    h=hashlib.sha256(); h.update(b"V131-V42-POLICY-CACHE-v1"); h.update(np.asarray([int(seed)],np.int64).tobytes()); h.update(tr.tobytes()); h.update(b"|"); h.update(va.tobytes())
    f=POLICY_CACHE/(h.hexdigest()+".npz")
    if f.exists():
        z=np.load(f); a=np.asarray(z["val_anchor"],np.float32); st=np.asarray(z["val_state"],np.int8)
        if a.shape==(len(va),M) and st.shape==(len(va),): return {"val_anchor":a,"val_state":st}
    z=build_policy_anchor_split(tr,va,int(seed)); np.savez_compressed(f,val_anchor=z["val_anchor"],val_state=z["val_state"])
    return {"val_anchor":np.asarray(z["val_anchor"],np.float32),"val_state":np.asarray(z["val_state"],np.int8)}
print("policy-anchor cache",POLICY_CACHE)


invalid assessment cells: 251 / 50000
administrative/noise rows: 535 / 14573
USER_FEATURES: (5000, 134)
example augmented text: [older] min aku mau fokus jadi data analyst aja deh mau kuasain excel sama sql tpl_93c96e6b92 topic_excel topic_sql [recent] eh min gajadi deh bosku nyuruh pindah haluan bikin ai ada kelas genai nggak tpl_abc6b29bac topic_genai recent_topic_genai recent_topic_genai [recent] sori nanya mulu ya min tapi kelas yg cover data cleaning ada silabus lengkapnya ga tpl_ace230065e topic_eda recent_topic_eda recent_topic_eda [current] sori nanya mulu ya min tapi kelas belajar pyton dari nol itu berapa lama 
catalog lexical matrices: (5000, 17) (5000, 17)
latest template groups: 1095
latest intent signatures train/test: 46 46
test intent signatures unseen in train: 0
readiness band counts: {np.int8(0): np.int64(45948), np.int8(1): np.int64(16376), np.int8(2): np.int64(5676)}
ordinal grade counts: {np.int16(0): np.int64(48009), np.int16(1): np.int64(1353), np.int16(2): np.i

In [4]:
# =========================
# 3. V129 deterministic views + target-blind validation geometry
# =========================
URL_RE=re.compile(r"https?://\S+|www\.\S+",re.I)
WS_RE=re.compile(r"\s+")
PUNCT_RE=re.compile(r"[^\w<>]+",re.UNICODE)

def norm_text(x):
    x="" if pd.isna(x) else str(x)
    x=html.unescape(unicodedata.normalize("NFKC",x)).casefold()
    x=URL_RE.sub(" <url> ",x); x=PUNCT_RE.sub(" ",x); return WS_RE.sub(" ",x).strip()

ALIASES=[
("generative ai",),("gen ai",),("genai",),("large language model",),("llm",),
("exploratory data analysis",),("eda",),("machine learning",),("ml",),
("computer vision",),("cv",),("natural language processing",),("nlp",),
("prompt engineering",),("web scraping",),("scraping",),("version control",),("git",),
("dashboard",),("mlops",),("no code ai",),("nocode",),("automation",),
("statistics",),("statistik",),("python",),("sql",),("excel",),("career",),("karir",),("portfolio",),("portofolio",)
]
ALIAS_PATTERNS=[re.compile(r"(?<!\w)"+re.escape(a[0])+r"(?!\w)") for a in sorted(ALIASES,key=lambda z:-len(z[0]))]
def skeleton(x):
    s=norm_text(x)
    for rx in ALIAS_PATTERNS: s=rx.sub(" <topic> ",s)
    s=re.sub(r"\b\d+\b"," <num> ",s); return WS_RE.sub(" ",s).strip()

chat=chat.copy(); chat["user_id"]=chat.user_id.astype(str)
chat["timestamp"]=pd.to_datetime(chat.timestamp,errors="coerce")
chat=chat.sort_values(["user_id","timestamp","chat_id"],kind="stable")
byu={u:g for u,g in chat.groupby("user_id",sort=False)}
full_text=[]; latest_text=[]; full_skel=[]; latest_skel=[]; msg_lists=[]; chat_count=[]
for u in ALL_USERS:
    g=byu.get(u)
    msgs=[] if g is None else [norm_text(t) for t in g.user_chat_text.tolist() if norm_text(t)]
    msg_lists.append(msgs); chat_count.append(len(msgs))
    latest=msgs[-1] if msgs else "<no_chat>"; full=" [sep] ".join(msgs) if msgs else "<no_chat>"
    latest_text.append(latest); full_text.append(full); latest_skel.append(skeleton(latest)); full_skel.append(skeleton(full))
full_text=np.array(full_text,dtype=object); latest_text=np.array(latest_text,dtype=object)
full_skel=np.array(full_skel,dtype=object); latest_skel=np.array(latest_skel,dtype=object); chat_count=np.asarray(chat_count,np.float32)

# Assessment parse: invalid/out-of-range values remain NaN until fold-local imputation.
WORD_NUM={"nol":0,"zero":0,"satu":1,"dua":2,"tiga":3,"empat":4,"lima":5}
def parse_obj(s):
    try: return json.loads(s)
    except Exception: return ast.literal_eval(s)
assess=assess.copy(); assess["user_id"]=assess.user_id.astype(str)
assert set(ALL_USERS).issubset(set(assess.user_id))
assess=assess.set_index("user_id").loc[ALL_USERS].reset_index()
objs=[parse_obj(s) for s in assess.assessment_result.astype(str)]
SKILLS=list(objs[0].keys()); assert len(SKILLS)==10
A_RAW=np.full((N,10),np.nan,np.float32); A_BAD=np.zeros((N,10),np.float32)
for i,o in enumerate(objs):
    for j,k in enumerate(SKILLS):
        v=o.get(k,np.nan); z=None
        if isinstance(v,str) and v.strip().casefold() in WORD_NUM: z=float(WORD_NUM[v.strip().casefold()])
        else:
            try: z=float(v)
            except Exception: z=np.nan
        if np.isfinite(z) and abs(z-round(z))<1e-8 and 0<=z<=5: A_RAW[i,j]=z
        else: A_BAD[i,j]=1.0
assert int(A_BAD.sum())==251, f"assessment parser drift: {A_BAD.sum()} invalid cells"

# Module documents and readiness map from organizer catalog semantics.
mc=modules.set_index("module_id").loc[MODULES]
MOD_TEXT=(mc.module_name.astype(str)+" [sep] "+mc.description_and_syllabus.astype(str)+" [sep] "+mc.prerequisite_level.astype(str)).map(norm_text).tolist()
MOD_SKILLS={
0:[],1:[0],2:[1],3:[0],4:[9],5:[2],6:[0,2,3],7:[1],8:[2,3,4,5],9:[4,6],10:[4],11:[4,7],12:[7],13:[7,8],14:[7],15:[0,4,5,9],16:[8,9]
}

# Target-blind near-template components. No-chat users are deliberately unique groups.
vec_grp=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=1,max_features=50000,norm="l2")
Xg=vec_grp.fit_transform(full_skel.tolist())
sim=(Xg@Xg.T).tocsr(); sim.data=(sim.data>=cfg.near_threshold).astype(np.int8); sim.eliminate_zeros()
_,near_group=connected_components(sim,directed=False)
nochat=np.array([t=="<no_chat>" for t in full_text])
base_gid=int(near_group.max())+1
near_group=near_group.astype(np.int64); near_group[nochat]=base_gid+np.flatnonzero(nochat)
ALL_TRAIN=np.arange(NTR,dtype=int)
ALL_TEST=np.arange(NTR,N,dtype=int)

# Manifest is target-free. It is diagnostic only and never used as a test label proxy.
pd.DataFrame({
    "user_id":train.user_id.astype(str),
    "near_group":near_group[:NTR],
    "latest_skeleton":latest_skel[:NTR],
    "full_history_sha256":[hashlib.sha256(str(t).encode()).hexdigest() for t in full_text[:NTR]],
}).to_csv(OUT/"v133_template_manifest.csv",index=False)
print("template geometry ready; near groups(train/all)=",len(np.unique(near_group[:NTR])),len(np.unique(near_group)))
del Xg,sim,vec_grp; gc.collect()


template geometry ready; near groups(train/all)= 1912 2297


0

In [5]:

# =========================
# 4. Frozen E5 + Nomic representations, pinned revisions and legacy-cache probe verification
# =========================
E5_ID="intfloat/multilingual-e5-base"; E5_REV="d128750597153bb5987e10b1c3493a34e5a4502a"
NOMIC_ID="nomic-ai/nomic-embed-text-v2-moe"; NOMIC_REV="1066b6599d099fbb93dfcb64f9c37a7c9e503e85"
NOMIC_CODE_REV="7710840340a098cfb869c4f65e87cf2b1b70caca"

def text_sha(texts,prefix,model_id):
    h=hashlib.sha256(); h.update((model_id+"|"+prefix).encode())
    for t in texts: h.update(b"\0"); h.update(str(t).encode("utf-8"))
    return h.hexdigest()

def _resolve_model(model_id):
    needle=model_id.split('/')[-1].lower().replace('_','-')
    if Path('/kaggle/input').exists():
        for p in Path('/kaggle/input').rglob('config.json'):
            s=str(p.parent).lower().replace('_','-')
            if needle in s: return str(p.parent)
    return model_id

def _external_pair(name,kind):
    if not Path('/kaggle/input').exists(): return None
    target=f"{name}_{kind}.npy"
    for f in Path('/kaggle/input').rglob(target):
        meta=f.with_suffix('.json')
        if meta.exists(): return f,meta
    return None

def _legacy_ok(pair,texts,prefix,model_id):
    if pair is None: return None
    f,meta=pair
    try:
        m=json.loads(meta.read_text()); arr=np.load(f,mmap_mode='r')
        legacy=text_sha(texts,prefix,model_id)
        if m.get('key')!=legacy or arr.shape[0]!=len(texts): return None
        return np.asarray(arr,np.float32)
    except Exception: return None

def _cos_rows(a,b):
    a=np.asarray(a,np.float32); b=np.asarray(b,np.float32)
    a=a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-12); b=b/np.maximum(np.linalg.norm(b,axis=1,keepdims=True),1e-12)
    return np.sum(a*b,axis=1)

def load_e5_pair(user_texts,module_texts,device):
    up=_legacy_ok(_external_pair('users','e5'),user_texts,'query: ',E5_ID)
    mp=_legacy_ok(_external_pair('modules','e5'),module_texts,'passage: ',E5_ID)
    import torch
    from sentence_transformers import SentenceTransformer
    mid=_resolve_model(E5_ID); model=SentenceTransformer(mid,device=device,revision=None if mid!=E5_ID else E5_REV)
    probes_u=[0,min(123,len(user_texts)-1),len(user_texts)-1]; probes_m=[0,min(5,len(module_texts)-1),len(module_texts)-1]
    if up is not None and mp is not None:
        pu=np.asarray(model.encode(['query: '+user_texts[i] for i in probes_u],batch_size=8,normalize_embeddings=True,convert_to_numpy=True),np.float32)
        pm=np.asarray(model.encode(['passage: '+module_texts[i] for i in probes_m],batch_size=8,normalize_embeddings=True,convert_to_numpy=True),np.float32)
        if float(_cos_rows(pu,up[probes_u]).min())>.999 and float(_cos_rows(pm,mp[probes_m]).min())>.999:
            print('Reusing probe-verified external E5 cache')
            del model; gc.collect(); torch.cuda.empty_cache(); return up,mp
    print('Encoding pinned E5 revision',E5_REV)
    up=np.asarray(model.encode(['query: '+t for t in user_texts],batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True),np.float32)
    mp=np.asarray(model.encode(['passage: '+t for t in module_texts],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True),np.float32)
    np.save(OUT/'users_e5.npy',up); np.save(OUT/'modules_e5.npy',mp)
    json.dump({'model':E5_ID,'revision':E5_REV,'user_key':text_sha(user_texts,'query: ',E5_ID),'module_key':text_sha(module_texts,'passage: ',E5_ID),'shape_users':list(up.shape),'shape_modules':list(mp.shape)},open(OUT/'e5_cache_meta.json','w'),indent=2)
    del model; gc.collect(); torch.cuda.empty_cache(); return up,mp

def _nomic_encode(model,tok,texts,prefix,device,batch=48):
    import torch
    rows=[]
    with torch.no_grad():
        for a in range(0,len(texts),batch):
            e=tok([prefix+t for t in texts[a:a+batch]],padding=True,truncation=True,max_length=256,return_tensors='pt'); e={k:v.to(device) for k,v in e.items()}
            h=model(**e).last_hidden_state; mask=e['attention_mask'].unsqueeze(-1).float()
            v=((h*mask).sum(1)/mask.sum(1).clamp_min(1)).float(); v=v/v.norm(dim=1,keepdim=True).clamp_min(1e-12); rows.append(v.cpu().numpy())
    return np.concatenate(rows).astype(np.float32)

def load_nomic_pair(user_texts,module_texts,device):
    up=_legacy_ok(_external_pair('users','nomic'),user_texts,'search_query: ',NOMIC_ID)
    mp=_legacy_ok(_external_pair('modules','nomic'),module_texts,'search_document: ',NOMIC_ID)
    import torch
    from transformers import AutoTokenizer,AutoModel
    mid=_resolve_model(NOMIC_ID)
    tok=AutoTokenizer.from_pretrained(mid,trust_remote_code=True,revision=None if mid!=NOMIC_ID else NOMIC_REV)
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    kw=dict(trust_remote_code=True,torch_dtype=dtype)
    if mid==NOMIC_ID: kw.update(revision=NOMIC_REV,code_revision=NOMIC_CODE_REV)
    model=AutoModel.from_pretrained(mid,**kw).to(device).eval()
    probes_u=[0,min(123,len(user_texts)-1),len(user_texts)-1]; probes_m=[0,min(5,len(module_texts)-1),len(module_texts)-1]
    if up is not None and mp is not None:
        pu=_nomic_encode(model,tok,[user_texts[i] for i in probes_u],'search_query: ',device,batch=8)
        pm=_nomic_encode(model,tok,[module_texts[i] for i in probes_m],'search_document: ',device,batch=8)
        if float(_cos_rows(pu,up[probes_u]).min())>.999 and float(_cos_rows(pm,mp[probes_m]).min())>.999:
            print('Reusing probe-verified external Nomic cache')
            del model; gc.collect(); torch.cuda.empty_cache(); return up,mp
    print('Encoding pinned Nomic revision',NOMIC_REV,'code',NOMIC_CODE_REV)
    up=_nomic_encode(model,tok,user_texts,'search_query: ',device); mp=_nomic_encode(model,tok,module_texts,'search_document: ',device,batch=17)
    np.save(OUT/'users_nomic.npy',up); np.save(OUT/'modules_nomic.npy',mp)
    json.dump({'model':NOMIC_ID,'revision':NOMIC_REV,'code_revision':NOMIC_CODE_REV,'user_key':text_sha(user_texts,'search_query: ',NOMIC_ID),'module_key':text_sha(module_texts,'search_document: ',NOMIC_ID),'shape_users':list(up.shape),'shape_modules':list(mp.shape)},open(OUT/'nomic_cache_meta.json','w'),indent=2)
    del model; gc.collect(); torch.cuda.empty_cache(); return up,mp

import torch
if not torch.cuda.is_available(): dev_e5=dev_no='cpu'; print('WARNING CUDA unavailable; production target is T4x2')
else:
    ng=torch.cuda.device_count(); dev_e5='cuda:0'; dev_no='cuda:1' if ng>1 else 'cuda:0'
E5U,E5M=load_e5_pair(full_text.tolist(),MOD_TEXT,dev_e5)
NOU,NOM=load_nomic_pair(full_text.tolist(),MOD_TEXT,dev_no)
assert E5U.shape==(N,768) and E5M.shape[0]==M and NOU.shape[0]==N and NOM.shape[0]==M
E5_MOD=(E5U@E5M.T).astype(np.float32); NOM_MOD=(NOU@NOM.T).astype(np.float32)
print('dense ready',E5U.shape,NOU.shape)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

configuration_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.90G [00:00<?, ?B/s]

Encoding pinned Nomic revision 1066b6599d099fbb93dfcb64f9c37a7c9e503e85 code 7710840340a098cfb869c4f65e87cf2b1b70caca
dense ready (5000, 768) (5000, 768)


In [6]:
# =========================
# 5. Lean V129 candidate features with explicit retrieval_anchor geometry
# =========================
def fit_bundle(ref_idx):
    ref_idx=np.asarray(ref_idx,int)
    corpus=[full_text[i] for i in ref_idx]+MOD_TEXT
    wv=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=cfg.word_max_features,sublinear_tf=True,smooth_idf=True,norm="l2")
    cv=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,max_features=cfg.char_max_features,sublinear_tf=True,norm="l2")
    wv.fit(corpus); cv.fit(corpus)
    return {
        "wv":wv,"cv":cv,
        "wfull":wv.transform(full_text.tolist()),"wlatest":wv.transform(latest_text.tolist()),"wm":wv.transform(MOD_TEXT),
        "cfull":cv.transform(full_text.tolist()),"clatest":cv.transform(latest_text.tolist()),"cm":cv.transform(MOD_TEXT)
    }

def recency_similarity(bundle,indices,key="word"):
    indices=np.asarray(indices,int); out=np.zeros((len(indices),M),np.float32)
    vec=bundle["wv"] if key=="word" else bundle["cv"]
    mm=bundle["wm"] if key=="word" else bundle["cm"]
    for qi,g in enumerate(indices):
        msgs=msg_lists[g]
        if not msgs: continue
        S=(vec.transform(msgs)@mm.T).toarray().astype(np.float32)
        w=np.power(0.5,np.arange(len(msgs)-1,-1,-1,dtype=np.float32)); w/=w.sum()
        out[qi]=(w[:,None]*S).sum(0)
    return out

def geom_from_scores(S,prefix):
    S=np.asarray(S,np.float32); order=np.argsort(-S,axis=1,kind="stable"); ranks=np.empty_like(order)
    rr=np.arange(len(S))[:,None]; ranks[rr,order]=np.arange(1,M+1)[None,:]
    best=S.max(1,keepdims=True); fifth=np.take_along_axis(S,order[:,4:5],1)
    return [(prefix+"_score",S),(prefix+"_rank",ranks.astype(np.float32)/M),(prefix+"_gapbest",best-S),(prefix+"_gap5",S-fifth)]

def impute_assessment(ref_idx,q_idx):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int)
    med=np.nanmedian(A_RAW[ref_idx],axis=0); med=np.where(np.isfinite(med),med,2.5)
    A=np.where(np.isfinite(A_RAW[q_idx]),A_RAW[q_idx],med[None,:]).astype(np.float32)
    return A,A_BAD[q_idx].astype(np.float32)

def readiness(A,BAD):
    R=np.zeros((len(A),M),np.float32); I=np.zeros_like(R)
    for m,js in MOD_SKILLS.items():
        if js: R[:,m]=A[:,js].mean(1); I[:,m]=BAD[:,js].mean(1)
        else: R[:,m]=A.mean(1); I[:,m]=BAD.mean(1)
    return R,I

def candidate_features(bundle,ref_idx,q_idx,retrieval_anchor_q):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int); pieces=[]
    wf=(bundle["wfull"][q_idx]@bundle["wm"].T).toarray().astype(np.float32)
    wl=(bundle["wlatest"][q_idx]@bundle["wm"].T).toarray().astype(np.float32)
    wr=recency_similarity(bundle,q_idx,"word")
    cf=(bundle["cfull"][q_idx]@bundle["cm"].T).toarray().astype(np.float32)
    cl=(bundle["clatest"][q_idx]@bundle["cm"].T).toarray().astype(np.float32)
    cr=recency_similarity(bundle,q_idx,"char")
    for p,S in [
        ("tfidf_word_full",wf),("tfidf_word_latest",wl),("tfidf_word_recency",wr),
        ("tfidf_char_full",cf),("tfidf_char_latest",cl),("tfidf_char_recency",cr),
        ("e5_mod",E5_MOD[q_idx]),("nomic_mod",NOM_MOD[q_idx]),
    ]: pieces.extend(geom_from_scores(S,p))
    pieces += [("dense_e5_minus_nomic",E5_MOD[q_idx]-NOM_MOD[q_idx])]

    Aq,Bq=impute_assessment(ref_idx,q_idx); R,I=readiness(Aq,Bq)
    pieces += [("ready",R),("ready_invalid",I)]
    for j,s in enumerate(SKILLS):
        pieces.append((f"assess_{j:02d}",np.repeat(Aq[:,j:j+1],M,1)))
        pieces.append((f"assess_bad_{j:02d}",np.repeat(Bq[:,j:j+1],M,1)))
    pieces += [
        ("assess_mean",np.repeat(Aq.mean(1,keepdims=True),M,1)),
        ("assess_std",np.repeat(Aq.std(1,keepdims=True),M,1)),
        ("assess_invalid_count",np.repeat(Bq.sum(1,keepdims=True),M,1)),
        ("chat_count",np.repeat(chat_count[q_idx,None],M,1)),
        ("no_chat",np.repeat((chat_count[q_idx]==0).astype(np.float32)[:,None],M,1)),
    ]
    pieces.extend(geom_from_scores(retrieval_anchor_q,"retrieval_anchor"))

    names=[]; arrays=[]
    for n,a in pieces:
        a=np.asarray(a,np.float32); assert a.shape==(len(q_idx),M),(n,a.shape)
        names.append(n); arrays.append(a[:,:,None])
    X=np.concatenate(arrays,2).astype(np.float32)
    assert np.isfinite(X).all()
    # V129 contract: no BM25 and no target-derived KNN/template statistics.
    assert not any(("bm25" in n) or n.startswith("knn_") or n.startswith("tpl_") for n in names)
    return X,names


In [7]:
# =========================
# 6. V129 Stage-1: retrieval_anchor + per-module LightGBM
# =========================
def make_inner_splits(indices,seed):
    indices=np.asarray(indices,int); groups=near_group[indices]; strat=np.argmax(y[indices],1)
    try:
        sg=StratifiedGroupKFold(cfg.n_inner,shuffle=True,random_state=seed)
        return [(indices[a],indices[b]) for a,b in sg.split(np.zeros(len(indices)),strat,groups)]
    except Exception:
        gg=GroupKFold(cfg.n_inner)
        return [(indices[a],indices[b]) for a,b in gg.split(np.zeros(len(indices)),groups=groups)]

def _anchor_matrix(bundle,ref_idx,q_idx,med_ref=None):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int)
    if med_ref is None:
        med_ref=np.nanmedian(A_RAW[ref_idx],axis=0); med_ref=np.where(np.isfinite(med_ref),med_ref,2.5)
    Aq=np.where(np.isfinite(A_RAW[q_idx]),A_RAW[q_idx],med_ref[None,:]).astype(np.float32)
    B=A_BAD[q_idx].astype(np.float32)
    extra=np.c_[Aq,B,chat_count[q_idx],(chat_count[q_idx]==0).astype(np.float32)]
    return hstack([bundle["wfull"][q_idx],bundle["cfull"][q_idx],csr_matrix(extra)],format="csr")

def nested_anchor(bundle,train_idx,query_idx,inner_splits):
    train_idx=np.asarray(train_idx,int); query_idx=np.asarray(query_idx,int); pos={g:i for i,g in enumerate(train_idx)}
    oof_rel=np.zeros((len(train_idx),M),np.float32); oof_top=np.zeros_like(oof_rel)
    for f,(tr,va) in enumerate(inner_splits):
        med=np.nanmedian(A_RAW[tr],axis=0); med=np.where(np.isfinite(med),med,2.5)
        Xtr=_anchor_matrix(bundle,tr,tr,med); Xv=_anchor_matrix(bundle,tr,va,med)
        r=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,y[tr])
        t=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,np.eye(M,dtype=np.float32)[np.argmax(y[tr],1)])
        loc=np.array([pos[g] for g in va]); oof_rel[loc]=r.predict(Xv); oof_top[loc]=t.predict(Xv)
    med=np.nanmedian(A_RAW[train_idx],axis=0); med=np.where(np.isfinite(med),med,2.5)
    Xtr=_anchor_matrix(bundle,train_idx,train_idx,med); Xq=_anchor_matrix(bundle,train_idx,query_idx,med)
    r=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,y[train_idx])
    t=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,np.eye(M,dtype=np.float32)[np.argmax(y[train_idx],1)])
    oof=(.9*oof_rel+.1*oof_top).astype(np.float32); q=(.9*r.predict(Xq)+.1*t.predict(Xq)).astype(np.float32)
    return oof,q

def fit_lgb_stage1(Xtr,ytr,Xq):
    assert Xtr.ndim==3 and Xq.ndim==3 and Xtr.shape[1]==Xq.shape[1]==M and Xtr.shape[2]==Xq.shape[2]
    pred=np.zeros((len(Xq),M),np.float32)
    for m in range(M):
        vals=[]
        for seed in cfg.lgb_seeds:
            model=lgb.LGBMRegressor(
                objective="regression_l2",n_estimators=cfg.lgb_estimators,learning_rate=cfg.lgb_lr,
                num_leaves=cfg.lgb_num_leaves,max_depth=cfg.lgb_max_depth,min_child_samples=cfg.lgb_min_child,
                subsample=cfg.lgb_subsample,subsample_freq=1,colsample_bytree=cfg.lgb_colsample,
                reg_alpha=cfg.lgb_reg_alpha,reg_lambda=cfg.lgb_reg_lambda,
                random_state=int(seed+37*m),n_jobs=2,verbosity=-1,force_col_wise=True)
            model.fit(Xtr[:,m,:],ytr[:,m]); vals.append(model.predict(Xq[:,m,:]))
        pred[:,m]=np.mean(vals,axis=0)
    return pred

def fit_stage1_once(train_idx,query_idx,bundle,seed):
    """Train Stage-1 without using any query labels. `bundle` may include query text vocabulary only label-free."""
    train_idx=np.asarray(train_idx,int); query_idx=np.asarray(query_idx,int)
    inner=make_inner_splits(train_idx,seed+11)
    anchor_tr,anchor_q=nested_anchor(bundle,train_idx,query_idx,inner)
    Xtr,names=candidate_features(bundle,train_idx,train_idx,anchor_tr)
    Xq,nq=candidate_features(bundle,train_idx,query_idx,anchor_q); assert nq==names
    pred=fit_lgb_stage1(Xtr,y[train_idx],Xq)
    assert np.isfinite(pred).all()
    del Xtr; gc.collect()
    return {"retrieval_anchor_query":anchor_q,"fusion_query":pred,"raw_query":Xq,"names":names}

def make_world_splits(indices,world,seed=SEED):
    indices=np.asarray(indices,int)
    strat=np.array([f"{np.argmax(y[i])}_{int((y[i]>0).sum())}" for i in indices],object)
    cc=Counter(strat); strat=np.array([s if cc[s]>=cfg.n_outer else str(np.argmax(y[i])) for s,i in zip(strat,indices)],object)
    if world=="random":
        sk=StratifiedKFold(cfg.n_outer,shuffle=True,random_state=seed)
        return [(indices[a],indices[b]) for a,b in sk.split(np.zeros(len(indices)),strat)]
    if world in {"exact","latest","near"}:
        groups={"exact":full_text[indices],"latest":latest_skel[indices],"near":near_group[indices]}[world]
        try:
            sg=StratifiedGroupKFold(cfg.n_outer,shuffle=True,random_state=seed)
            return [(indices[a],indices[b]) for a,b in sg.split(np.zeros(len(indices)),strat,groups)]
        except Exception:
            gg=GroupKFold(cfg.n_outer)
            return [(indices[a],indices[b]) for a,b in gg.split(np.zeros(len(indices)),groups=groups)]
    if world=="contiguous":
        order=indices[np.argsort([int(re.sub(r"\D","",ALL_USERS[i])) for i in indices])]
        blocks=np.array_split(order,cfg.n_outer)
        return [(np.setdiff1d(indices,b),np.asarray(b,int)) for b in blocks]
    raise ValueError(world)


In [8]:
# =========================
# 7. Legacy V129/V130 analogue head + V132 hierarchical all-rank utility representation
# =========================
def action_table(indices,policy_anchor,fusion,raw,names):
    indices=np.asarray(indices,int); policy_anchor=np.asarray(policy_anchor,np.float32); n=len(indices)
    X=[]; Y=[]; users=[]; cand=[]; inc=[]; ranks=[]
    order=np.argsort(-policy_anchor,axis=1,kind='stable')
    for i,g in enumerate(indices):
        im=int(order[i,0])
        for rr in (1,2,3):
            cm=int(order[i,rr]); s=policy_anchor[i].copy()
            if g<NTR:
                before=per_user_ndcg(y[g:g+1],s[None,:])[0]; s[im],s[cm]=s[cm],s[im]
                delta=float(per_user_ndcg(y[g:g+1],s[None,:])[0]-before)
            else: delta=0.0
            diff=raw[i,cm]-raw[i,im]
            pair=np.zeros(M*M,np.float32); pair[im*M+cm]=1.0
            c1=np.zeros(M,np.float32); c1[cm]=1.0
            i1=np.zeros(M,np.float32); i1[im]=1.0
            r1=np.zeros(3,np.float32); r1[rr-1]=1.0
            feat=np.r_[fusion[i,cm],fusion[i,im],fusion[i,cm]-fusion[i,im],policy_anchor[i,im]-policy_anchor[i,cm],diff,c1,i1,pair,r1]
            X.append(feat); Y.append(delta); users.append(i); cand.append(cm); inc.append(im); ranks.append(rr+1)
    fn=['fusion_c','fusion_i','fusion_diff','policy_anchor_margin']+['d_'+x for x in names]+[f'cand_{m}' for m in MODULES]+[f'inc_{m}' for m in MODULES]+[f'pair_{a}_{b}' for a in MODULES for b in MODULES]+['rank2','rank3','rank4']
    X=np.asarray(X,np.float32); Y=np.asarray(Y,np.float32)
    assert X.shape[0]==3*n and np.isfinite(X).all() and np.isfinite(Y).all()
    return X,Y,np.asarray(users,int),np.asarray(cand,int),np.asarray(inc,int),np.asarray(ranks,int),fn

def fit_action_model(X,Y):
    model=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver='lsqr',tol=1e-5,max_iter=3000)); model.fit(X,Y); return model

def predict_action(model,indices,policy_anchor,fusion,raw,names):
    X,Y,U,C,I,R,fn=action_table(indices,policy_anchor,fusion,raw,names)
    return {'X':X,'y':Y,'user':U,'cand':C,'inc':I,'rank':R,'pred':model.predict(X).astype(np.float32),'features':fn}

def bootstrap_ci(d,groups=None,seed=SEED):
    d=np.asarray(d,float); rng=np.random.default_rng(seed); vals=np.empty(cfg.bootstraps,float)
    if groups is None:
        n=len(d)
        for b in range(cfg.bootstraps): vals[b]=d[rng.integers(0,n,n)].mean()
    else:
        groups=np.asarray(groups); ug=np.unique(groups); bucket={g:np.flatnonzero(groups==g) for g in ug}
        for b in range(cfg.bootstraps):
            pick=rng.choice(ug,len(ug),replace=True); ix=np.concatenate([bucket[g] for g in pick]); vals[b]=d[ix].mean()
    return float(np.quantile(vals,.025)),float(np.quantile(vals,.975))

def choose_user_actions(pred,y_delta,cand,inc,n_users):
    P=np.asarray(pred).reshape(n_users,3); D=np.asarray(y_delta).reshape(n_users,3); C=np.asarray(cand).reshape(n_users,3); I=np.asarray(inc).reshape(n_users,3)
    j=np.argmax(P,1); rr=np.arange(n_users); return P[rr,j],D[rr,j],C[rr,j],I[rr,j],j

def select_fixed_cap(score,realized,candidate,eligible,cap):
    score=np.asarray(score,float); realized=np.asarray(realized,float); eligible=np.asarray(eligible,bool)
    ids=np.flatnonzero((score>0)&eligible); pick=ids[np.argsort(-score[ids],kind='stable')[:min(int(cap),len(ids))]]
    sel=np.zeros(len(score),bool); sel[pick]=True; ud=np.zeros(len(score),float); ud[pick]=realized[pick]
    return sel,ud,pick

def categorical_tv(a,b):
    a=pd.Series(list(a),dtype='object').value_counts(normalize=True); b=pd.Series(list(b),dtype='object').value_counts(normalize=True)
    idx=a.index.union(b.index); return float(0.5*np.abs(a.reindex(idx,fill_value=0)-b.reindex(idx,fill_value=0)).sum())

def smd(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float); den=np.sqrt((np.var(a)+np.var(b))/2.0)
    return float((np.mean(b)-np.mean(a))/max(den,1e-12))

assert abs(ndcg5(y[:5],y[:5])-1.0)<1e-8
print('legacy action utility helpers ready')

# -------------------------
# V132 lexical complement: exact V128 Okapi parameters, full-history views only.
# -------------------------
class BM25Index:
    def __init__(self,docs,ngram=(1,1),k1=1.2,b=0.75,max_features=70000):
        self.k1=float(k1); self.b=float(b)
        self.vec=CountVectorizer(token_pattern=r"(?u)\b\w+\b",ngram_range=ngram,max_features=max_features)
        c=self.vec.fit_transform(list(docs)).tocsr().astype(np.float32)
        self.n=c.shape[0]
        df=np.asarray((c>0).sum(0)).ravel()
        self.idf=np.log(1+(self.n-df+.5)/(df+.5)).astype(np.float32)
        dl=np.asarray(c.sum(1)).ravel(); avg=max(float(dl.mean()),1e-6)
        d=c.copy()
        for i in range(self.n):
            a,z=d.indptr[i],d.indptr[i+1]
            tf,cols=d.data[a:z],d.indices[a:z]
            den=tf+self.k1*(1-self.b+self.b*dl[i]/avg)
            d.data[a:z]=tf*(self.k1+1)/np.maximum(den,1e-12)*self.idf[cols]
        self.d=d
    def score(self,queries):
        return np.asarray((self.vec.transform(list(queries)).astype(np.float32)@self.d.T).toarray(),np.float32)

BM25_UNI=BM25Index(MOD_TEXT,(1,1),1.2,0.75,70000)
BM25_BI=BM25Index(MOD_TEXT,(1,2),1.2,0.75,70000)
BM25_UNI_FULL=BM25_UNI.score(full_text.tolist())
BM25_BI_FULL=BM25_BI.score(full_text.tolist())
assert np.isfinite(BM25_UNI_FULL).all() and np.isfinite(BM25_BI_FULL).all()

NEG_RX=re.compile(r"\b(?:tidak|ga|gak|nggak|enggak|bukan|jangan|tanpa|belum|skip|hapus|cancel|batal)\b",re.I)
COMP_RX=re.compile(r"\b(?:atau|vs|versus|dibanding|daripada|mending|lebih cocok|bedanya|beda)\b",re.I)
NEG_FLAG=np.asarray([bool(NEG_RX.search(str(t))) for t in full_text],np.float32)
COMP_FLAG=np.asarray([bool(COMP_RX.search(str(t))) for t in full_text],np.float32)
TOPIC_PRESENT=(MODULE_COUNT>0).astype(np.float32)
LATEST_TOPIC_PRESENT=(MODULE_LATEST>0).astype(np.float32)
LATEST_FULL_CONTRADICTION=np.asarray([
    bool(TOPIC_PRESENT[i].any() and LATEST_TOPIC_PRESENT[i].any() and np.any(TOPIC_PRESENT[i]!=LATEST_TOPIC_PRESENT[i]))
    for i in range(N)
],np.float32)

def _nranks(S):
    S=np.asarray(S,np.float32); order=np.argsort(-S,axis=1,kind='stable'); ranks=np.empty_like(order)
    ranks[np.arange(len(S))[:,None],order]=np.arange(1,M+1)[None,:]
    return ranks.astype(np.float32)/M

def _global_view_matrices(bundle,q_idx):
    q_idx=np.asarray(q_idx,int)
    return {
        'tfidf_word_full':(bundle['wfull'][q_idx]@bundle['wm'].T).toarray().astype(np.float32),
        'tfidf_word_latest':(bundle['wlatest'][q_idx]@bundle['wm'].T).toarray().astype(np.float32),
        'tfidf_word_recency':recency_similarity(bundle,q_idx,'word'),
        'tfidf_char_full':(bundle['cfull'][q_idx]@bundle['cm'].T).toarray().astype(np.float32),
        'tfidf_char_latest':(bundle['clatest'][q_idx]@bundle['cm'].T).toarray().astype(np.float32),
        'tfidf_char_recency':recency_similarity(bundle,q_idx,'char'),
        'e5':E5_MOD[q_idx],'nomic':NOM_MOD[q_idx],
        'bm25_uni_full':BM25_UNI_FULL[q_idx],'bm25_bi_full':BM25_BI_FULL[q_idx],
    }

def global_action_table(indices,policy_anchor,bundle,ref_idx):
    """V132 primary row contract: global pooling, explicit policy margin, no pair one-hot, full-history BM25 only."""
    indices=np.asarray(indices,int); ref_idx=np.asarray(ref_idx,int); anchor=np.asarray(policy_anchor,np.float32); n=len(indices)
    order=np.argsort(-anchor,axis=1,kind='stable')
    inc=np.repeat(order[:,0],3); cand=order[:,1:4].reshape(-1); rank=np.tile(np.array([2,3,4],np.int8),n)
    grow=np.repeat(indices,3); qloc=np.repeat(np.arange(n),3)

    c1=np.eye(M,dtype=np.float32)[cand]; i1=np.eye(M,dtype=np.float32)[inc]; r1=np.eye(3,dtype=np.float32)[rank-2]
    margin=(anchor[qloc,inc]-anchor[qloc,cand]).reshape(-1,1).astype(np.float32)
    parts=[c1,i1,r1,margin]
    names=[f'cand_{m}' for m in MODULES]+[f'inc_{m}' for m in MODULES]+['rank2','rank3','rank4','policy_anchor_margin']

    views=_global_view_matrices(bundle,indices)
    for key in ['tfidf_word_full','tfidf_word_latest','tfidf_word_recency','tfidf_char_full','tfidf_char_latest','tfidf_char_recency','e5','nomic','bm25_uni_full','bm25_bi_full']:
        S=views[key]; R=_nranks(S); cc=S[qloc,cand]; ii=S[qloc,inc]
        parts.append(np.c_[cc,ii,cc-ii,R[qloc,cand],R[qloc,inc]].astype(np.float32))
        names += [key+'_c',key+'_i',key+'_diff',key+'_rank_c',key+'_rank_i']

    Aq,Bq=impute_assessment(ref_idx,indices); ready_q,_=readiness(Aq,Bq)
    rc=ready_q[qloc,cand]; ri=ready_q[qloc,inc]
    state=np.c_[
        rc,ri,rc-ri,np.repeat(Aq.mean(1),3),np.repeat(Aq.std(1),3),np.repeat(Bq.sum(1),3),
        np.repeat(np.log1p(chat_count[indices]),3),np.repeat((chat_count[indices]==0).astype(np.float32),3),
        np.repeat(NEG_FLAG[indices],3),np.repeat(COMP_FLAG[indices],3),np.repeat(LATEST_FULL_CONTRADICTION[indices],3),
        TOPIC_PRESENT[grow,cand],TOPIC_PRESENT[grow,inc]
    ].astype(np.float32)
    parts.append(state); names += ['ready_c','ready_i','ready_diff','assess_mean','assess_std','assess_invalid_count','log_chat_count','no_chat','negation','comparison','latest_full_contradiction','candidate_explicit','incumbent_explicit']

    X=np.c_[*parts].astype(np.float32); Y=np.zeros(3*n,np.float32)
    for i,g in enumerate(indices):
        if g>=NTR: continue
        base=anchor[i].copy(); before=per_user_ndcg(y[g:g+1],base[None,:])[0]
        for j in range(3):
            s=base.copy(); a=int(order[i,0]); b=int(order[i,j+1]); s[a],s[b]=s[b],s[a]
            Y[3*i+j]=float(per_user_ndcg(y[g:g+1],s[None,:])[0]-before)
    assert X.shape[0]==3*n and len(names)==X.shape[1] and np.isfinite(X).all() and np.isfinite(Y).all()
    assert not any(x.startswith('pair_') for x in names)
    assert 'policy_anchor_margin' in names and 'bm25_uni_full_diff' in names and 'bm25_bi_full_diff' in names
    assert not any(('bm25_uni_latest' in x) or ('bm25_uni_recency' in x) or ('bm25_bi_latest' in x) for x in names)
    return {'X':X,'y':Y,'inc':inc.astype(np.int16),'cand':cand.astype(np.int16),'rank':rank,'features':names}

def feature_mask(names,variant='primary'):
    names=list(names); keep=np.ones(len(names),bool)
    if variant=='primary': return keep
    if variant=='no_bm25':
        return np.array([not x.startswith('bm25_') for x in names],bool)
    raise ValueError(variant)

def _pair_rank_key(inc,cand,rank):
    return np.asarray(inc,int)*(M*3)+np.asarray(cand,int)*3+(np.asarray(rank,int)-2)

def fit_hierarchical_action(table,variant='primary',extra_X=None,extra_names=None):
    X=np.asarray(table['X'],np.float32); names=list(table['features'])
    if extra_X is not None:
        X=np.c_[X,np.asarray(extra_X,np.float32)]; names += list(extra_names)
    mask=feature_mask(names,variant) if extra_X is None else np.ones(len(names),bool)
    model=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver='lsqr',tol=1e-5,max_iter=3000))
    model.fit(X[:,mask],table['y'])
    base=model.predict(X[:,mask]).astype(np.float32)
    resid=np.asarray(table['y'],np.float32)-base
    keys=_pair_rank_key(table['inc'],table['cand'],table['rank'])
    support={}; adjust={}
    for k in np.unique(keys):
        z=(keys==k); n=int(z.sum()); support[int(k)]=n; adjust[int(k)]=float(resid[z].sum()/(n+cfg.pair_rank_shrink))
    return {'model':model,'mask':mask,'support':support,'adjust':adjust,'names':names,'variant':variant}

def predict_hierarchical(head,table,extra_X=None):
    X=np.asarray(table['X'],np.float32)
    if extra_X is not None: X=np.c_[X,np.asarray(extra_X,np.float32)]
    base=head['model'].predict(X[:,head['mask']]).astype(np.float32)
    keys=_pair_rank_key(table['inc'],table['cand'],table['rank'])
    adj=np.asarray([head['adjust'].get(int(k),0.0) for k in keys],np.float32)
    sup=np.asarray([head['support'].get(int(k),0) for k in keys],np.int16)
    return (base+adj).reshape(-1,3),base.reshape(-1,3),adj.reshape(-1,3),sup.reshape(-1,3)

def select_allrank_fixed_cap(pred3,y3,eligible,cap):
    P=np.asarray(pred3,np.float32); D=np.asarray(y3,np.float32).reshape(-1,3); eligible=np.asarray(eligible,bool)
    j=np.argmax(P,1); rr=np.arange(len(P)); score=P[rr,j]; realized=D[rr,j]
    ids=np.flatnonzero((score>0)&eligible); pick=ids[np.argsort(-score[ids],kind='stable')[:min(int(cap),len(ids))]]
    sel=np.zeros(len(P),bool); sel[pick]=True; ud=np.zeros(len(P),float); ud[pick]=realized[pick]
    sj=np.full(len(P),-1,np.int8); sj[pick]=j[pick].astype(np.int8)
    return sel,ud,pick,score,sj

def select_rank2_fixed_cap(pred3,y3,eligible,cap):
    P=np.asarray(pred3,np.float32); D=np.asarray(y3,np.float32).reshape(-1,3); score=P[:,0]; realized=D[:,0]
    ids=np.flatnonzero((score>0)&np.asarray(eligible,bool)); pick=ids[np.argsort(-score[ids],kind='stable')[:min(int(cap),len(ids))]]
    sel=np.zeros(len(P),bool); sel[pick]=True; ud=np.zeros(len(P),float); ud[pick]=realized[pick]
    return sel,ud,pick,score

def fusion_action_block(table,fusion):
    fusion=np.asarray(fusion,np.float32); n=len(fusion)
    C=table['cand'].reshape(n,3); I=table['inc'].reshape(n,3)
    fc=np.take_along_axis(fusion,C,1).reshape(-1); fi=np.take_along_axis(fusion,I,1).reshape(-1)
    return np.c_[fc,fi,fc-fi].astype(np.float32),['diag_fusion_c','diag_fusion_i','diag_fusion_diff']

def fold_prefix_cap(f,total):
    q,r=divmod(int(total),cfg.n_outer); return q+(1 if int(f)<r else 0)

print('V132 upstream reconstruction helpers ready; alpha',cfg.action_alpha,'pair-rank shrink',cfg.pair_rank_shrink)


legacy action utility helpers ready
V132 upstream reconstruction helpers ready; alpha 10.0 pair-rank shrink 20.0


In [9]:
# =========================
# 8. V133 third-level nested V132 analogue + three fixed expert heads
# =========================
def largest_remainder_quotas(total,sizes):
    sizes=np.asarray(sizes,float); raw=float(total)*sizes/max(float(sizes.sum()),1.0); q=np.floor(raw).astype(int)
    left=int(total)-int(q.sum()); order=np.lexsort((np.arange(len(sizes)),-(raw-q)))
    for i in order[:left]: q[i]+=1
    return q

def _apply_user_swaps(scores,selected,inc,cand):
    out=np.asarray(scores,np.float32).copy()
    for i in np.flatnonzero(selected):
        a=int(inc[i]); b=int(cand[i]); out[i,a],out[i,b]=out[i,b],out[i,a]
    return out

def fit_v132_stack_predict(train_idx,query_idx,seed):
    """Rebuild V129->V130->V132 authority using train_idx labels only; query labels never enter fitting."""
    train_idx=np.asarray(train_idx,int); query_idx=np.asarray(query_idx,int)
    bundle=fit_bundle(train_idx); inner=make_inner_splits(train_idx,int(seed)+11); pos={g:i for i,g in enumerate(train_idx)}
    ftr=np.zeros((len(train_idx),M),np.float32); ranchor_tr=np.zeros_like(ftr); panchor_tr=np.zeros_like(ftr); rtr=None; names=None
    for j,(ir,iv) in enumerate(inner):
        ib=fit_bundle(ir)
        z=fit_stage1_once(ir,iv,ib,int(seed)+1000+37*j)
        pa=cached_policy_anchor_split(ir,iv,int(seed)+2000+53*j)
        loc=np.array([pos[g] for g in iv]); ftr[loc]=z['fusion_query']; ranchor_tr[loc]=z['retrieval_anchor_query']; panchor_tr[loc]=pa['val_anchor']
        if rtr is None:
            rtr=np.zeros((len(train_idx),M,z['raw_query'].shape[2]),np.float32); names=z['names']
        assert z['names']==names; rtr[loc]=z['raw_query']
        del ib,z,pa; gc.collect()
    assert np.isfinite(ftr).all() and np.isfinite(ranchor_tr).all() and np.isfinite(panchor_tr).all() and np.isfinite(rtr).all()

    X129,Y129,*_,fn129=action_table(train_idx,ranchor_tr,ftr,rtr,names); m129=fit_action_model(X129,Y129)
    X130,Y130,*_,fn130=action_table(train_idx,panchor_tr,ftr,rtr,names); m130=fit_action_model(X130,Y130); assert fn129==fn130
    gtr=global_action_table(train_idx,panchor_tr,bundle,train_idx); h132=fit_hierarchical_action(gtr,'primary')

    zq=fit_stage1_once(train_idx,query_idx,bundle,int(seed)+5000)
    paq=cached_policy_anchor_split(train_idx,query_idx,int(seed)+6000)
    a129=predict_action(m129,query_idx,zq['retrieval_anchor_query'],zq['fusion_query'],zq['raw_query'],zq['names'])
    a130=predict_action(m130,query_idx,paq['val_anchor'],zq['fusion_query'],zq['raw_query'],zq['names'])
    nq=len(query_idx)
    s129,d129,c129,i129,j129=choose_user_actions(a129['pred'],a129['y'],a129['cand'],a129['inc'],nq)
    ss129,_,_=select_fixed_cap(s129,d129,c129,(paq['val_state']==0),int(math.ceil(cfg.v129_coverage*nq)))
    s130,d130,c130,i130,j130=choose_user_actions(a130['pred'],a130['y'],a130['cand'],a130['inc'],nq)
    e130=(paq['val_state']==0)&(~ss129)
    ss130,_,_=select_fixed_cap(s130,d130,c130,e130,int(math.ceil(cfg.v130_coverage*nq)))
    ev132=(paq['val_state']==0)&(~ss129)&(~ss130)
    gq=global_action_table(query_idx,paq['val_anchor'],bundle,train_idx)
    p132,_,_,_=predict_hierarchical(h132,gq)
    ss132,_,_,_,sj132=select_allrank_fixed_cap(p132,gq['y'],ev132,int(math.ceil(cfg.v132_coverage*nq)))

    base=np.asarray(paq['val_anchor'],np.float32).copy()
    base=_apply_user_swaps(base,ss129,i129,c129)
    base=_apply_user_swaps(base,ss130,i130,c130)
    C132=gq['cand'].reshape(nq,3); I132=gq['inc'].reshape(nq,3)
    c132=np.zeros(nq,np.int16); i132=np.zeros(nq,np.int16)
    for i in np.flatnonzero(ss132):
        j=int(sj132[i]); c132[i]=C132[i,j]; i132[i]=I132[i,j]
    base=_apply_user_swaps(base,ss132,i132,c132)
    eligible=(paq['val_state']==0)&(~ss129)&(~ss130)&(~ss132)
    out={
        'base':base,'eligible':eligible,'policy_state':paq['val_state'],'v129_selected':ss129,'v130_selected':ss130,'v132_selected':ss132,
        'fusion':zq['fusion_query'],'query_bundle':bundle
    }
    del X129,Y129,X130,Y130,m129,m130,gtr,h132,zq,paq,a129,a130,gq,rtr,ftr,ranchor_tr,panchor_tr
    gc.collect(); return out

COMMON_NAMES=None
STATE_NAMES=['ready_c','ready_i','ready_diff','assess_mean','assess_std','assess_invalid_count','log_chat_count','no_chat','negation','comparison','latest_full_contradiction','candidate_explicit','incumbent_explicit']
SPARSE_DIFF_NAMES=['tfidf_word_full_diff','tfidf_word_latest_diff','tfidf_word_recency_diff','tfidf_char_full_diff','tfidf_char_latest_diff','tfidf_char_recency_diff']
BM25_DIFF_NAMES=['bm25_uni_full_diff','bm25_bi_full_diff']

def v133_table(indices,base_scores,bundle,ref_idx):
    tab=global_action_table(indices,base_scores,bundle,ref_idx); names=tab['features']; mp={n:i for i,n in enumerate(names)}
    common=[n for n in names if n.startswith('cand_') or n.startswith('inc_') or n in {'rank2','rank3','rank4','policy_anchor_margin'}]
    assert all(n in mp for n in SPARSE_DIFF_NAMES+BM25_DIFF_NAMES+STATE_NAMES)
    spec={
        'sparse':common+SPARSE_DIFF_NAMES,
        'retrieval':common+BM25_DIFF_NAMES,
        'state':common+STATE_NAMES,
    }
    X={k:tab['X'][:,[mp[n] for n in vv]].astype(np.float32) for k,vv in spec.items()}
    return tab,X,spec

def fit_v133_experts(table,Xdict,user_eligible):
    row_ok=np.repeat(np.asarray(user_eligible,bool),3); assert row_ok.sum()>0
    models={}; pred_train={}
    for k,X in Xdict.items():
        model=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver='lsqr',tol=1e-5,max_iter=3000))
        model.fit(X[row_ok],table['y'][row_ok]); models[k]=model; pred_train[k]=model.predict(X).astype(np.float32)
    keys=_pair_rank_key(table['inc'],table['cand'],table['rank']); support={}
    for kk in np.unique(keys[row_ok]): support[int(kk)]=int(np.sum(keys[row_ok]==kk))
    return models,pred_train,support

def predict_v133_experts(models,Xdict):
    return {k:models[k].predict(Xdict[k]).astype(np.float32) for k in ('sparse','retrieval','state')}

def action_support_array(table,support):
    keys=_pair_rank_key(table['inc'],table['cand'],table['rank'])
    return np.asarray([support.get(int(k),0) for k in keys],np.int16)

def consensus_policy(table,preds,support,user_eligible,cap):
    P=np.c_[preds['sparse'],preds['retrieval'],preds['state']].astype(np.float32)
    mean=P.mean(1); votes=(P>0).sum(1).astype(np.int8); sup=action_support_array(table,support)
    admitted=(mean>0)&(votes>=2)&(sup>=cfg.v133_support_min)&np.repeat(np.asarray(user_eligible,bool),3)
    n=len(user_eligible); M3=mean.reshape(n,3); A3=admitted.reshape(n,3); V3=votes.reshape(n,3); S3=sup.reshape(n,3)
    j=np.full(n,-1,np.int8); uscore=np.full(n,-np.inf,np.float32)
    for i in range(n):
        q=np.flatnonzero(A3[i])
        if len(q):
            # stable argmax: candidate rank order resolves exact ties.
            jj=int(q[np.argmax(M3[i,q])]); j[i]=jj; uscore[i]=M3[i,jj]
    ids=np.flatnonzero(np.isfinite(uscore)&(uscore>0)&np.asarray(user_eligible,bool))
    pick=ids[np.argsort(-uscore[ids],kind='stable')[:min(int(cap),len(ids))]]
    sel=np.zeros(n,bool); sel[pick]=True; ud=np.zeros(n,float)
    yy=table['y'].reshape(n,3)
    for i in pick: ud[i]=float(yy[i,int(j[i])])
    return {'selected':sel,'pick':pick,'chosen_j':j,'user_score':uscore,'user_delta':ud,'mean_action':mean.reshape(n,3),'votes_action':votes.reshape(n,3),'support_action':sup.reshape(n,3),'raw_action':P.reshape(n,3,3)}

def _ref_smd(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    if len(a)==0 or len(b)==0: return float('nan')
    if len(b)<2:
        den=max(float(np.std(a)),1e-12)
    else:
        den=max(float(np.sqrt((np.var(a)+np.var(b))/2.0)),1e-12)
    return float((np.mean(b)-np.mean(a))/den)

print('V133 third-level helpers ready; experts=sparse/retrieval/state; alpha',cfg.action_alpha,'support floor',cfg.v133_support_min)


V133 third-level helpers ready; experts=sparse/retrieval/state; alpha 10.0 support floor 10


In [10]:
# =========================
# 9. Fully third-level nested V133 primary random world
# =========================
def fully_nested_v133_primary(indices):
    indices=np.asarray(indices,int); local={g:i for i,g in enumerate(indices)}; n=len(indices)
    wseed=cfg.seed+133; outer=make_world_splits(indices,'random',wseed)
    sizes=[len(va) for _,va in outer]
    quotas={t:largest_remainder_quotas(t,sizes) for t in (cfg.v133_primary_actions,*cfg.v133_sensitivity_actions)}
    fold_id=np.full(n,-1,int); eligible=np.zeros(n,bool); selected=np.zeros(n,bool); chosen_j=np.full(n,-1,np.int8); user_delta=np.zeros(n,float)
    user_score=np.full(n,-np.inf,np.float32); action_y=np.zeros((n,3),np.float32); action_c=np.zeros((n,3),np.int16); action_i=np.zeros((n,3),np.int16); action_rank=np.zeros((n,3),np.int8)
    action_mean=np.zeros((n,3),np.float32); action_votes=np.zeros((n,3),np.int8); action_support=np.zeros((n,3),np.int16); action_raw=np.zeros((n,3,3),np.float32)
    base_scores=np.zeros((n,M),np.float32); v132_analogue_selected=np.zeros(n,bool)
    prefix_ud={t:np.zeros(n,float) for t in cfg.v133_sensitivity_actions}
    Xoof={}; Xnames=None; yflat=np.zeros(3*n,np.float32); row_eligible=np.zeros(3*n,bool); globalX=None; global_names=None
    fold_rows=[]; ledger=[]

    for f,(tr,va) in enumerate(outer):
        t0=time.time(); pos={g:i for i,g in enumerate(tr)}
        # Third level: each outer-training row receives a V132 analogue produced without its labels and without outer-val labels.
        inner=make_inner_splits(tr,wseed+1000+f)
        base_tr=np.zeros((len(tr),M),np.float32); elig_tr=np.zeros(len(tr),bool); v132sel_tr=np.zeros(len(tr),bool)
        for j,(ir,iv) in enumerate(inner):
            z=fit_v132_stack_predict(ir,iv,wseed+10000*f+100*j)
            loc=np.array([pos[g] for g in iv]); base_tr[loc]=z['base']; elig_tr[loc]=z['eligible']; v132sel_tr[loc]=z['v132_selected']
            del z; gc.collect()
        assert np.isfinite(base_tr).all()
        zv=fit_v132_stack_predict(tr,va,wseed+50000+f)
        base_va=zv['base']; elig_va=zv['eligible']
        bundle=fit_bundle(tr)
        ttr,Xtr,spec=v133_table(tr,base_tr,bundle,tr)
        tv,Xv,specv=v133_table(va,base_va,bundle,tr); assert specv==spec and tv['features']==ttr['features']
        models,_,support=fit_v133_experts(ttr,Xtr,elig_tr)
        pv=predict_v133_experts(models,Xv)
        cap=int(quotas[cfg.v133_primary_actions][f]); pol=consensus_policy(tv,pv,support,elig_va,cap)
        lv=np.array([local[g] for g in va]); fold_id[lv]=f; eligible[lv]=elig_va; selected[lv]=pol['selected']; chosen_j[lv]=pol['chosen_j']; user_delta[lv]=pol['user_delta']; user_score[lv]=pol['user_score']
        base_scores[lv]=base_va; v132_analogue_selected[lv]=zv['v132_selected']
        action_y[lv]=tv['y'].reshape(len(va),3); action_c[lv]=tv['cand'].reshape(len(va),3); action_i[lv]=tv['inc'].reshape(len(va),3); action_rank[lv]=tv['rank'].reshape(len(va),3)
        action_mean[lv]=pol['mean_action']; action_votes[lv]=pol['votes_action']; action_support[lv]=pol['support_action']; action_raw[lv]=pol['raw_action']

        # Persist fold-pure expert rows for final full-train action heads.
        if Xnames is None:
            Xnames=spec; global_names=tv['features']; globalX=np.zeros((3*n,tv['X'].shape[1]),np.float32)
            for k in Xv: Xoof[k]=np.zeros((3*n,Xv[k].shape[1]),np.float32)
        assert Xnames==spec and global_names==tv['features']
        for q,li in enumerate(lv):
            sl=slice(3*li,3*li+3); sv=slice(3*q,3*q+3); yflat[sl]=tv['y'][sv]; row_eligible[sl]=elig_va[q]; globalX[sl]=tv['X'][sv]
            for k in Xoof: Xoof[k][sl]=Xv[k][sv]

        # Fixed sensitivity prefixes use identical gates and fold-local experts; never change the registered primary.
        for total in cfg.v133_sensitivity_actions:
            pp=consensus_policy(tv,pv,support,elig_va,int(quotas[total][f])); prefix_ud[total][lv]=pp['user_delta']

        sel=pol['selected']; yy=pol['user_delta'][sel]
        fold_rows.append({'fold':f,'n':len(va),'v132_analogue_n':int(zv['v132_selected'].sum()),'residual_eligible_n':int(elig_va.sum()),'quota':cap,'selected_n':int(sel.sum()),'gain':float(pol['user_delta'].mean()),'benefit':int((yy>0).sum()),'harm':int((yy<0).sum()),'tie':int((yy==0).sum()),'seconds':time.time()-t0})
        C=tv['cand'].reshape(len(va),3); I=tv['inc'].reshape(len(va),3); R=tv['rank'].reshape(len(va),3)
        for q,g in enumerate(va):
            jj=int(pol['chosen_j'][q]);
            if jj<0: continue
            ledger.append({'user_id':ALL_USERS[g],'fold':f,'eligible':bool(elig_va[q]),'selected':bool(sel[q]),'incumbent':MODULES[int(I[q,jj])],'challenger':MODULES[int(C[q,jj])],'candidate_rank':int(R[q,jj]),'consensus_mean':float(pol['mean_action'][q,jj]),'sparse_pred':float(pol['raw_action'][q,jj,0]),'retrieval_pred':float(pol['raw_action'][q,jj,1]),'state_pred':float(pol['raw_action'][q,jj,2]),'votes':int(pol['votes_action'][q,jj]),'pair_rank_support':int(pol['support_action'][q,jj]),'realized_delta':float(tv['y'].reshape(len(va),3)[q,jj])})
        del zv,bundle,ttr,tv,Xtr,Xv,models,pv,support; gc.collect()

    assert (fold_id>=0).all() and np.isfinite(base_scores).all() and np.isfinite(globalX).all()
    fg=[float(user_delta[fold_id==f].mean()) for f in range(cfg.n_outer)]
    ciu=bootstrap_ci(user_delta,seed=SEED+133); cig=bootstrap_ci(user_delta,near_group[indices],seed=SEED+134)
    return {'indices':indices,'fold_id':fold_id,'eligible':eligible,'selected':selected,'chosen_j':chosen_j,'user_delta':user_delta,'user_score':user_score,'gain':float(user_delta.mean()),'fold_gains':fg,'ci_user':ciu,'ci_cluster':cig,'action_y':action_y,'action_cand':action_c,'action_inc':action_i,'action_rank':action_rank,'action_mean':action_mean,'action_votes':action_votes,'action_support':action_support,'action_raw':action_raw,'base_scores':base_scores,'v132_analogue_selected':v132_analogue_selected,'prefix_ud':prefix_ud,'fold_metrics':pd.DataFrame(fold_rows),'ledger':pd.DataFrame(ledger),'Xoof':Xoof,'Xspec':Xnames,'globalX':globalX,'global_names':global_names,'yflat':yflat,'row_eligible':row_eligible}

primary=fully_nested_v133_primary(ALL_TRAIN)
primary['fold_metrics'].to_csv(OUT/'v133_primary_fold_metrics.csv',index=False)
primary['ledger'].to_csv(OUT/'v133_primary_action_ledger.csv',index=False)
print('V133 primary gain',primary['gain'],'folds',primary['fold_gains'],'CI',primary['ci_user'],primary['ci_cluster'])
display(primary['fold_metrics'])


V133 primary gain 0.00010068836947903037 folds [0.000120464488863945, 0.00020117569947615267, 0.00022508100606501104, 1.2104790657758713e-05, -5.538413766771555e-05] CI (4.499486449640244e-06, 0.00022014888036064794) (-2.0218665145835913e-05, 0.0002466342172659617)


,fold,n,v132_analogue_n,residual_eligible_n,quota,selected_n,gain,benefit,harm,tie,seconds
0,0,800,4,636,3,3,0.000120,3,0,0,757.874802
1,1,800,4,637,3,3,0.000201,3,0,0,743.523606
2,2,800,4,637,2,2,0.000225,2,0,0,740.049136
3,3,800,4,573,2,2,0.000012,1,1,0,751.377608
4,4,800,4,554,2,2,-0.000055,1,1,0,746.444412


In [11]:
# =========================
# 10. V133 diagnostics, prefix/slice stability, and promotion evidence
# =========================
sel=primary['selected']; ud=primary['user_delta']; selected_y=ud[sel]
benefit=int((selected_y>0).sum()); harm=int((selected_y<0).sum()); tie=int((selected_y==0).sum())
pos_folds=int(sum(x>0 for x in primary['fold_gains']))
prefix_rows=[{'actions':cfg.v133_primary_actions,'gain':primary['gain'],'selected_n':int(sel.sum())}]
for total,z in primary['prefix_ud'].items():
    prefix_rows.append({'actions':int(total),'gain':float(z.mean()),'selected_n':int(np.count_nonzero(z))})
prefix_df=pd.DataFrame(prefix_rows).sort_values('actions'); prefix_df.to_csv(OUT/'v133_prefix_metrics.csv',index=False); display(prefix_df)

# Target-blind / representation slices on the registered OOF package. These diagnose contradictions; they never retune the policy.
core_names=['policy_anchor_margin','tfidf_word_full_diff','tfidf_char_full_diff','bm25_uni_full_diff','bm25_bi_full_diff','ready_diff','log_chat_count','no_chat','negation','latest_full_contradiction','candidate_explicit']
mp={n:i for i,n in enumerate(primary['global_names'])}; assert all(n in mp for n in core_names)
selected_rows=[]
for i in np.flatnonzero(sel):
    j=int(primary['chosen_j'][i]); selected_rows.append(3*i+j)
selected_rows=np.asarray(selected_rows,int)
core_oof=primary['globalX'][selected_rows][:,[mp[n] for n in core_names]] if len(selected_rows) else np.zeros((0,len(core_names)),np.float32)

slice_rows=[]
def _slice(name,mask):
    mask=np.asarray(mask,bool)&sel; vals=ud[mask]
    slice_rows.append({'slice':name,'n':int(mask.sum()),'package_contribution':float(vals.sum()/NTR),'mean_if_selected':float(vals.mean()) if len(vals) else np.nan,'benefit':int((vals>0).sum()),'harm':int((vals<0).sum())})
_slice('rank2',sel&(primary['chosen_j']==0)); _slice('rank3',sel&(primary['chosen_j']==1)); _slice('rank4',sel&(primary['chosen_j']==2))
_slice('no_chat',sel&(chat_count[:NTR]==0)); _slice('has_chat',sel&(chat_count[:NTR]>0))
_slice('expert_vote_3of3',sel&(np.max(primary['action_votes'],axis=1)>=3)); _slice('expert_vote_2of3',sel&(np.max(primary['action_votes'],axis=1)==2))
# Deployment-relevant world slices: whether each held-out user regime was represented in that fold's training partition.
exact_seen=np.zeros(NTR,bool); latest_seen=np.zeros(NTR,bool); near_seen=np.zeros(NTR,bool)
for f in range(cfg.n_outer):
    va=np.flatnonzero(primary['fold_id']==f); tr=np.flatnonzero(primary['fold_id']!=f)
    ex=set(full_text[tr]); la=set(latest_skel[tr]); ng=set(near_group[tr].tolist())
    exact_seen[va]=[full_text[i] in ex for i in va]; latest_seen[va]=[latest_skel[i] in la for i in va]; near_seen[va]=[int(near_group[i]) in ng for i in va]
for name,arr in [('exact_seen',exact_seen),('latest_seen',latest_seen),('near_seen',near_seen)]:
    _slice(name,arr); _slice(name.replace('_seen','_unseen'),~arr)
for b,idx in enumerate(np.array_split(np.arange(NTR),5)):
    m=np.zeros(NTR,bool); m[idx]=True; _slice(f'contiguous_block_{b}',m)
slice_df=pd.DataFrame(slice_rows); slice_df.to_csv(OUT/'v133_slice_metrics.csv',index=False); display(slice_df)
pd.DataFrame({'user_id':train.user_id.astype(str),'fold':primary['fold_id'],'eligible_v133':primary['eligible'],'selected_v133':primary['selected']}).to_csv(OUT/'v133_split_manifest.csv',index=False)
# A contradiction is material only if a prespecified slice with >=3 selected actions can erase the whole package by itself.
slice_contradiction=bool(((slice_df['n']>=3)&(slice_df['package_contribution']<=-abs(primary['gain']))).any())

np.savez_compressed(OUT/'v133_primary_oof.npz',fold_id=primary['fold_id'],eligible=primary['eligible'],selected=primary['selected'],chosen_j=primary['chosen_j'],user_delta=primary['user_delta'],action_y=primary['action_y'],action_cand=primary['action_cand'],action_inc=primary['action_inc'],action_rank=primary['action_rank'],action_mean=primary['action_mean'],action_votes=primary['action_votes'],action_support=primary['action_support'])
(OUT/'v133_expert_feature_names.json').write_text(json.dumps(primary['Xspec'],indent=2))
print('selected',int(sel.sum()),'benefit/harm/tie',benefit,harm,tie,'positive folds',pos_folds,'slice contradiction',slice_contradiction)


,actions,gain,selected_n
0,12,0.000101,12
1,20,0.000090,20
2,40,0.000118,39


,slice,n,package_contribution,mean_if_selected,benefit,harm
0,rank2,12,0.000101,0.033563,10,2
1,rank3,0,0.000000,NaN,0,0
2,rank4,0,0.000000,NaN,0,0
3,no_chat,2,0.000015,0.029384,2,0
4,has_chat,10,0.000086,0.034399,8,2
5,expert_vote_3of3,12,0.000101,0.033563,10,2
6,expert_vote_2of3,0,0.000000,NaN,0,0
7,exact_seen,6,0.000077,0.051247,6,0
8,exact_unseen,6,0.000024,0.015879,4,2
9,latest_seen,6,0.000077,0.051247,6,0


selected 12 benefit/harm/tie 10 2 0 positive folds 4 slice contradiction False


In [12]:
# =========================
# 11. Final V133 fit on clean OOF action rows -> exact V132 test base
# =========================
# Final expert heads are fitted on fold-pure OOF features/labels, never on in-sample V132 geometry.
row_ok=np.asarray(primary['row_eligible'],bool); yflat=np.asarray(primary['yflat'],np.float32)
final_models={}
for k,X in primary['Xoof'].items():
    m=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver='lsqr',tol=1e-5,max_iter=3000))
    m.fit(X[row_ok],yflat[row_ok]); final_models[k]=m
# Pair-by-rank support is counted only from residual-eligible fold-pure train opportunities.
incflat=primary['action_inc'].reshape(-1); candflat=primary['action_cand'].reshape(-1); rankflat=primary['action_rank'].reshape(-1)
keys=_pair_rank_key(incflat,candflat,rankflat); final_support={}
for kk in np.unique(keys[row_ok]): final_support[int(kk)]=int(np.sum(keys[row_ok]==kk))

bundle_all=fit_bundle(ALL_TRAIN); test_idx=ALL_TEST
# Exact public V132 bytes define deployment candidates and margins.
ttest,Xtest,spec_test=v133_table(test_idx,V132_BASE_SCORES,bundle_all,ALL_TRAIN); assert spec_test==primary['Xspec']
ptest=predict_v133_experts(final_models,Xtest)
ptest_policy=consensus_policy(ttest,ptest,final_support,V133_TEST_ELIGIBLE,cfg.v133_test_cap)
tsel=ptest_policy['selected']; trows=np.flatnonzero(tsel); actual_actions=int(tsel.sum())
C=ttest['cand'].reshape(NTE,3); I=ttest['inc'].reshape(NTE,3); R=ttest['rank'].reshape(NTE,3)

# Test opportunity ledger: all user-best gated proposals, selected or not.
ledger=[]
for i,u in enumerate(test.user_id.astype(str)):
    j=int(ptest_policy['chosen_j'][i])
    if j<0:
        ledger.append({'user_id':u,'eligible_residual':bool(V133_TEST_ELIGIBLE[i]),'selected_v133':False,'incumbent':'','challenger':'','candidate_rank':np.nan,'consensus_mean':np.nan,'sparse_pred':np.nan,'retrieval_pred':np.nan,'state_pred':np.nan,'votes':0,'pair_rank_support':0})
        continue
    ledger.append({'user_id':u,'eligible_residual':bool(V133_TEST_ELIGIBLE[i]),'selected_v133':bool(tsel[i]),'incumbent':MODULES[int(I[i,j])],'challenger':MODULES[int(C[i,j])],'candidate_rank':int(R[i,j]),'consensus_mean':float(ptest_policy['mean_action'][i,j]),'sparse_pred':float(ptest_policy['raw_action'][i,j,0]),'retrieval_pred':float(ptest_policy['raw_action'][i,j,1]),'state_pred':float(ptest_policy['raw_action'][i,j,2]),'votes':int(ptest_policy['votes_action'][i,j]),'pair_rank_support':int(ptest_policy['support_action'][i,j])})
test_ledger=pd.DataFrame(ledger); test_ledger.to_csv(OUT/'v133_test_action_opportunity_ledger.csv',index=False)
sel_ledger=test_ledger[test_ledger.selected_v133].copy(); sel_ledger.to_csv(OUT/'v133_selected_test_actions.csv',index=False); display(sel_ledger)

# Shift diagnostics use the exact chosen action rows and the same core feature contract as OOF.
test_selected_rows=np.asarray([3*i+int(ptest_policy['chosen_j'][i]) for i in trows],int)
core_test=ttest['X'][test_selected_rows][:,[ttest['features'].index(n) for n in core_names]] if len(test_selected_rows) else np.zeros((0,len(core_names)),np.float32)
core_smd={n:_ref_smd(core_oof[:,j],core_test[:,j]) for j,n in enumerate(core_names)} if len(core_test) else {n:np.nan for n in core_names}
oof_scores=primary['user_score'][primary['selected']]; test_scores=ptest_policy['user_score'][tsel]
utility_smd=_ref_smd(oof_scores,test_scores) if len(test_scores) else np.nan
max_abs_smd=max([abs(v) for v in list(core_smd.values())+[utility_smd] if np.isfinite(v)],default=0.0)

oof_pairs=[]
for i in np.flatnonzero(primary['selected']):
    j=int(primary['chosen_j'][i]); oof_pairs.append(f"{MODULES[int(primary['action_inc'][i,j])]}>{MODULES[int(primary['action_cand'][i,j])]}")
test_pairs=[f"{r.incumbent}>{r.challenger}" for r in sel_ledger.itertuples(index=False)]
max_pair_count=max(Counter(test_pairs).values(),default=0)
unseen_pair_rate=float(np.mean([p not in set(oof_pairs) for p in test_pairs])) if test_pairs else 0.0
selected_ids=set(sel_ledger.user_id.astype(str)); v131_overlap=len(selected_ids & V131_SELECTED_TEST_IDS)
all_test_action_gates=bool((actual_actions==0) or ((sel_ledger.consensus_mean>0).all() and (sel_ledger.votes>=2).all() and (sel_ledger.pair_rank_support>=cfg.v133_support_min).all()))
# Full final train support table for audit; no target means/labels are used in deployment gating.
support_rows=[]
for kk,nn in sorted(final_support.items()):
    inc=int(kk//(M*3)); rem=int(kk%(M*3)); cand=int(rem//3); rank=int(rem%3)+2
    support_rows.append({'incumbent':MODULES[inc],'challenger':MODULES[cand],'candidate_rank':rank,'support':int(nn)})
pd.DataFrame(support_rows).to_csv(OUT/'v133_pair_rank_support.csv',index=False)

# Byte-preserving candidate: only selected rows are changed, and only by swapping existing score strings.
def mutate_v132_bytes(base_bytes,selected_rows,chosen_j):
    text=base_bytes.decode('utf-8'); final_nl=text.endswith('\n'); lines=text.splitlines(); header=lines[0].split(','); col={m:header.index(m) for m in MODULES}
    ids=test.user_id.astype(str).tolist(); changed=set()
    for i in selected_rows:
        j=int(chosen_j[i]); a=MODULES[int(I[i,j])]; b=MODULES[int(C[i,j])]; parts=lines[i+1].split(',')
        parts[col[a]],parts[col[b]]=parts[col[b]],parts[col[a]]; lines[i+1]=','.join(parts); changed.add(i)
    out='\n'.join(lines)+('\n' if final_nl else '')
    return out.encode('utf-8'),changed

cand_bytes,changed_rows=mutate_v132_bytes(V132_BASE_BYTES,trows,ptest_policy['chosen_j'])
(OUT/'candidate_v133_DIAGNOSTIC.csv').write_bytes(cand_bytes)
candidate=pd.read_csv(io.BytesIO(cand_bytes)); CANDIDATE_SCORES=candidate[MODULES].to_numpy(np.float32)
# Exact geometry and byte-level immutability outside intended rows.
ba=top5_idx(V132_BASE_SCORES); bb=top5_idx(CANDIDATE_SCORES); changed_top1=(ba[:,0]!=bb[:,0])
base_lines=V132_BASE_BYTES.decode().splitlines(); cand_lines=cand_bytes.decode().splitlines()
immutable_rows_exact=all(base_lines[i+1]==cand_lines[i+1] for i in range(NTE) if i not in changed_rows)
geometry_audit={
    'base_sha256':hashlib.sha256(V132_BASE_BYTES).hexdigest(),'candidate_sha256':hashlib.sha256(cand_bytes).hexdigest(),'new_actions':actual_actions,
    'changed_top1':int(changed_top1.sum()),'all_changes_selected':bool(set(np.flatnonzero(changed_top1))==set(trows)),'immutable_unedited_rows_exact':bool(immutable_rows_exact),
    'protected_v132_rows_exact':bool(all(base_lines[i+1]==cand_lines[i+1] for i in np.flatnonzero(V132_TEST_SELECTED))),
    'locked_noneligible_exact':bool(all(base_lines[i+1]==cand_lines[i+1] for i in np.flatnonzero(~V133_TEST_ELIGIBLE))),
    'top5_membership_unchanged':bool(all(set(a)==set(b) for a,b in zip(ba,bb))),
    'rank5_unchanged':bool(np.array_equal(ba[:,4],bb[:,4])),
    'tail_rank5_to17_order_unchanged':bool(np.array_equal(np.argsort(-V132_BASE_SCORES,axis=1,kind='stable')[:,4:],np.argsort(-CANDIDATE_SCORES,axis=1,kind='stable')[:,4:])),
    'score_multiset_preserved':bool(all(np.array_equal(np.sort(V132_BASE_SCORES[i]),np.sort(CANDIDATE_SCORES[i])) for i in trows)),
    'finite':bool(np.isfinite(CANDIDATE_SCORES).all()),'row_order_unchanged':bool(candidate.user_id.astype(str).tolist()==test.user_id.astype(str).tolist())
}
(OUT/'v133_geometry_audit.json').write_text(json.dumps(geometry_audit,indent=2))
print('test actions',actual_actions,'utility SMD',utility_smd,'max abs SMD',max_abs_smd,'pairs',Counter(test_pairs),'unseen pair rate',unseen_pair_rate,'V131 overlap',v131_overlap)


,user_id,eligible_residual,selected_v133,incumbent,challenger,candidate_rank,consensus_mean,sparse_pred,retrieval_pred,state_pred,votes,pair_rank_support
135,U_4136,True,True,M_001,M_015,2.0,0.030488,0.017602,0.017299,0.056563,3,15
352,U_4353,True,True,M_001,M_015,2.0,0.022811,0.010943,0.010756,0.046733,3,15
589,U_4590,True,True,M_001,M_015,2.0,0.027215,0.014515,0.014266,0.052865,3,15


test actions 3 utility SMD 2.3017953011810537 max abs SMD 3.16227766016838 pairs Counter({'M_001>M_015': 3}) unseen pair rate 0.0 V131 overlap 2


In [13]:
# =========================
# 12. Frozen V133 decision gate + fail-close submission
# =========================
primary_gain=float(primary['gain']); pos_folds=int(sum(x>0 for x in primary['fold_gains']))
primary_selected=primary['user_delta'][primary['selected']]; benefits=int((primary_selected>0).sum()); harms=int((primary_selected<0).sum()); ties=int((primary_selected==0).sum())
prefix20=float(primary['prefix_ud'][20].mean())

hard_checks={
    'base_sha_ok':hashlib.sha256(V132_BASE_BYTES).hexdigest()==V132_BASE_SHA256,
    'oof_complete':bool((primary['fold_id']>=0).all()),
    'finite_oof':bool(np.isfinite(primary['globalX']).all() and np.isfinite(primary['action_mean']).all()),
    'geometry_ok':bool(all(geometry_audit[k] for k in ['all_changes_selected','immutable_unedited_rows_exact','protected_v132_rows_exact','locked_noneligible_exact','top5_membership_unchanged','rank5_unchanged','tail_rank5_to17_order_unchanged','score_multiset_preserved','finite','row_order_unchanged'])),
    'test_action_count_le_3':actual_actions<=cfg.v133_test_cap,
    'test_action_gates':all_test_action_gates,
    'pseudo_gt_used':False,
}
correctness_ok=bool(all(v for k,v in hard_checks.items() if k!='pseudo_gt_used') and hard_checks['pseudo_gt_used'] is False)
promotion={
    'gain_ge_0_00015':primary_gain>=cfg.submit_gain,
    'positive_folds_ge_4':pos_folds>=cfg.min_positive_folds,
    'benefit_ge_8':benefits>=cfg.min_benefit,
    'harm_le_4':harms<=cfg.max_harm,
    'prefix20_positive':prefix20>0,
    'test_actions_1_to_3':1<=actual_actions<=cfg.v133_test_cap,
    'selected_test_support_and_votes':all_test_action_gates,
    'max_abs_smd_le_1_5':max_abs_smd<=cfg.max_smd,
    'not_complete_v131_overlap':not (actual_actions==3 and v131_overlap==3),
    'no_material_slice_contradiction':not slice_contradiction,
}
submit_ok=bool(correctness_ok and all(promotion.values()))
if submit_ok: status='SUBMIT'
elif correctness_ok and primary_gain>0 and pos_folds>=3: status='BORDERLINE'
else: status='DO NOT SUBMIT'

candidate_bytes=(OUT/'candidate_v133_DIAGNOSTIC.csv').read_bytes()
if status=='SUBMIT':
    (OUT/'submission_v133_primary.csv').write_bytes(candidate_bytes)
else:
    (OUT/'submission_v133_primary.csv').write_bytes(V132_BASE_BYTES)
primary_sha=hashlib.sha256((OUT/'submission_v133_primary.csv').read_bytes()).hexdigest(); candidate_sha=hashlib.sha256(candidate_bytes).hexdigest()
if status!='SUBMIT': assert primary_sha==V132_BASE_SHA256

report={
    'version':'V133','status':status,'recommend_submit':bool(status=='SUBMIT'),'protected_public_lb':V132_PUBLIC_LB,'protected_base_sha256':V132_BASE_SHA256,
    'primary_incremental_gain':primary_gain,'positive_random_folds':pos_folds,'fold_gains':primary['fold_gains'],'ci_user':primary['ci_user'],'ci_cluster':primary['ci_cluster'],
    'primary_actions':int(primary['selected'].sum()),'benefit':benefits,'harm':harms,'tie':ties,'prefix20_gain':prefix20,'prefix40_gain':float(primary['prefix_ud'][40].mean()),
    'test_new_actions':actual_actions,'test_max_pair_count':max_pair_count,'test_unseen_pair_rate':unseen_pair_rate,'utility_smd_oof_to_test_selected':utility_smd,'core_smd':core_smd,'max_abs_smd':max_abs_smd,'v131_overlap':v131_overlap,
    'hard_checks':hard_checks,'promotion_gates':promotion,'geometry_audit':geometry_audit,'candidate_sha256':candidate_sha,'primary_sha256':primary_sha,
    'submission_instruction':('SUBMIT submission_v133_primary.csv' if status=='SUBMIT' else 'HOLD exact V132; candidate_v133_DIAGNOSTIC.csv is diagnostic only.')
}
(OUT/'v133_decision_report.json').write_text(json.dumps(report,indent=2))
versions={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'sklearn':__import__('sklearn').__version__,'lightgbm':lgb.__version__,'seed':SEED,'v132_base_sha256':V132_BASE_SHA256,'e5_model':E5_ID,'e5_revision':E5_REV,'nomic_model':NOMIC_ID,'nomic_revision':NOMIC_REV,'nomic_code_revision':NOMIC_CODE_REV}
(OUT/'v133_package_versions.json').write_text(json.dumps(versions,indent=2))
print(json.dumps(report,indent=2))
print('Artifacts:',sorted(p.name for p in OUT.iterdir() if p.is_file()))


{
  "version": "V133",
  "status": "BORDERLINE",
  "recommend_submit": false,
  "protected_public_lb": 0.66219,
  "protected_base_sha256": "d89714241c85dbaf93a5574550a5d9272c37b9b432c69fd5c5a2ff07fe690d84",
  "primary_incremental_gain": 0.00010068836947903037,
  "positive_random_folds": 4,
  "fold_gains": [
    0.000120464488863945,
    0.00020117569947615267,
    0.00022508100606501104,
    1.2104790657758713e-05,
    -5.538413766771555e-05
  ],
  "ci_user": [
    4.499486449640244e-06,
    0.00022014888036064794
  ],
  "ci_cluster": [
    -2.0218665145835913e-05,
    0.0002466342172659617
  ],
  "primary_actions": 12,
  "benefit": 10,
  "harm": 2,
  "tie": 0,
  "prefix20_gain": 8.994328835979104e-05,
  "prefix40_gain": 0.00011826139595359563,
  "test_new_actions": 3,
  "test_max_pair_count": 3,
  "test_unseen_pair_rate": 0.0,
  "utility_smd_oof_to_test_selected": 2.3017953011810537,
  "core_smd": {
    "policy_anchor_margin": -0.9551808300245214,
    "tfidf_word_full_diff": -0.748456

## Expected artifacts

The notebook writes the registered V133 validation evidence, fold/action ledgers, prefix/slice diagnostics, test opportunity ledger, feature contract, OOF NPZ, geometry/provenance reports, `candidate_v133_DIAGNOSTIC.csv`, and `submission_v133_primary.csv`. The latter is byte-identical V132 unless every frozen promotion gate passes.
